# Fog density and object detection

**Question:** does adding a fog-density estimate help detect objects in fog, while hurting clear-image accuracy?

Compare YOLO11n, YOLO11m and DETR, each with RGB control and predicted density. Detection means finding **labeled objects and their bounding boxes**, not merely classifying an image as foggy.

The existing training profiles stay independent. `train` resumes that profile; `analyze` redraws saved results; **`evaluate` uses existing trained weights on matched clear/fog images without retraining**. This is one standalone notebook; all implementation is ordinary Python inside collapsed notebook cells.

## 1. Choose your profile and run settings

Select the profile, version, data location, and output settings below. The notebook always prepares and validates the selected data. It loads the profile's saved fog-density predictor or trains and saves one when missing, then resumes detector checkpoints or reuses completed results. Set `retrain=True` only to start a separate fresh detector run.

The matched clear/fog comparison has its own optional cell after training. It reads labeled matched Cityscapes data and saved detector checkpoints into a separate output folder; run it independently whenever the selected detector profile is already trained.


In [ ]:
# fmt: off
#@title Shared constants and profile settings (expand to edit versions)
# fmt: on
from copy import deepcopy
from pathlib import Path
import os
import subprocess
import sys

# Shared names and defaults. Rename artifacts or models here once.
NOTEBOOK_REVISION = "standalone-20261012"
ENGINE_VERSION = 5  # Existing checkpoint payload format.
DEFAULT_SEED = 42
FULL_RUN_SEEDS = [DEFAULT_SEED, DEFAULT_SEED + 1, DEFAULT_SEED + 2]
YOLO_SMALL, YOLO_MEDIUM, DETR_MODEL = "YOLO11n", "YOLO11m", "DETR"
MODEL_SPECS = {
    YOLO_SMALL: "yolo11n.pt",
    YOLO_MEDIUM: "yolo11m.pt",
    DETR_MODEL: "facebook/detr-resnet-50",
}
ARM_RGB, ARM_DENSITY, ARM_ORACLE = "rgb_control", "predicted_density", "oracle_density"
DETECTION_ARMS = [ARM_RGB, ARM_DENSITY]
ARMS = [*DETECTION_ARMS, ARM_ORACLE]
PROFILE_SETUP, PROFILE_2K, PROFILE_5K, PROFILE_FULL = (
    "setup",
    "partial-2k",
    "partial-5k",
    "full",
)
PROFILE_NAMES = (PROFILE_SETUP, PROFILE_2K, PROFILE_5K, PROFILE_FULL)
SETUP_RUN_NAME, FULL_RUN_NAME = "setup-check-1", "colab-full-study"
DENSITY_RUN_PREFIX = "density"
DATA_FOLDER = "fog-study"
DRIVE_MOUNT, DRIVE_SUBDIRECTORY = "/content/drive", "MyDrive"
DATA_ARCHIVE = "fog-colab-data.zip"
CITYSCAPES_ARCHIVE = "Cityscape-Dataset.zip"
DATA_ARCHIVE_URL = (
    "https://drive.google.com/uc?export=download&id=14fNOg5ysWcZyll6oW3Nr28ROmFnWSexU"
)
MANIFEST_PATHS = {
    "manifest": "data/fog/nuscenes_all.json",
    "city_manifest": "data/fog/cityscapes.json",
}
CLASSES = ["person", "car", "truck", "bus", "train", "motorcycle", "bicycle"]
COCO_IDS, YOLO_IDS = [1, 3, 8, 6, 7, 4, 2], [0, 2, 7, 5, 6, 3, 1]
BETA_SCALE = 0.02
DENSITY_INPUT_SIZE = 128
BENCHMARK_IMAGES, BENCHMARK_BATCH_SIZE = 2000, 64
DENSITY_BIN_NAMES = ("clear", "light", "moderate", "dense")
DENSITY_BIN_UPPER_BOUNDS = (0.0, 0.005, 0.01, BETA_SCALE)
PROFILE_SELECTION_SCHEMA = 1
STUDY_IMAGE_TIMESTAMP_NS = 946684800000000000
VALID_EVALUATION_SPLITS = {"val", "test"}
BEST_CHECKPOINT, LAST_CHECKPOINT = "best.pt", "last.pt"
RUN_SETTINGS_FILE, INPUT_MANIFEST_FILE = "settings.json", "input-manifest.json"
INPUT_SELECTION_FILE = "input-selection.json"
PROGRESS_FILE, LOSS_HISTORY_FILE = "progress.json", "history.csv"
RESULTS_FILE, STATUS_FILE, STATUS_TABLE_FILE = (
    "results.csv",
    "status.json",
    "status.csv",
)
PAIRED_DELTA_FILE, DENSITY_REPORT_FILE = "paired-deltas.csv", "density-quality.csv"
MODEL_EXPORT_FILE, RUN_LOCK_FILE, FIGURE_DIRECTORY = (
    "trained-model.pt",
    "run.lock",
    "figures",
)
SEED_SUMMARY_FILE, HYPOTHESIS_FILE = "seed-summary.csv", "fog-vs-clear.csv"
HYPOTHESIS_SUMMARY_FILE = "hypothesis-summary.csv"
TRAINING_HEALTH_FILE, DATASET_AUDIT_FILE = "training-health.csv", "dataset-audit.csv"
CLASS_COVERAGE_FILE = "class-coverage.csv"
DATASET_AUDIT_FIGURE = "dataset-audit.png"
DENSITY_DIAGNOSTIC_FIGURE = "density-diagnostics.png"
DENSITY_ESTIMATES_FILE = "density-estimates.csv"
LABELED_CITYSCAPES_FILE = "cityscapes-detector.json"
TRAINING_DEFAULTS = {
    "density_epochs": 15,
    "density_checkpoint": None,
    "density_sampling": "natural",
    "batch_size": 1,
    "effective_batch_size": 4,
    "accumulate": 4,
    "loader": {"num_workers": 0, "prefetch_factor": None, "pin_memory": False},
    "repeats": 1,
    "learning_rates": {YOLO_SMALL: 1e-4, YOLO_MEDIUM: 1e-4, DETR_MODEL: 1e-5},
    "adapter_lr": 1e-3,
}
FULL_RUN_DEFAULTS = {
    key: TRAINING_DEFAULTS[key]
    for key in ("batch_size", "effective_batch_size", "loader", "density_sampling")
}
FULL_RUN_IMMUTABLE_KEYS = tuple(TRAINING_DEFAULTS) + (
    "epochs",
    "seeds",
    "arms",
    "size",
    "save_every",
    "amp",
    "device",
    "checkpoints",
)
PROFILE_COMMON = {
    "seed": DEFAULT_SEED,
    "models": list(MODEL_SPECS),
    "size": 640,
    "save_every": 200,
    "evaluation_split": "val",
    "early_stopping_patience": 2,
    "overwrite_existing": False,
    "density_sampling": "natural",
}
PROFILE_SETTINGS = {
    PROFILE_SETUP: {
        **PROFILE_COMMON,
        "training_image_cap": 48,
        "epochs": 1,
        "size": 320,
        "save_every": 2,
        "early_stopping_patience": 0,
        "overwrite_existing": True,
        "run_name": SETUP_RUN_NAME,
    },
    PROFILE_2K: {
        **PROFILE_COMMON,
        "training_image_cap": 2000,
        "epochs": 5,
        "version": "v5",
        "run_name": PROFILE_2K,
    },
    PROFILE_5K: {
        **PROFILE_COMMON,
        "training_image_cap": 5000,
        "epochs": 5,
        "version": "v6",
        "run_name": PROFILE_5K,
    },
    PROFILE_FULL: {
        **PROFILE_COMMON,
        "training_image_cap": None,
        "epochs": 30,
        "evaluation_split": "test",
        "early_stopping_patience": 0,
        "version": "v1",
        "run_name": FULL_RUN_NAME,
    },
}
FOG_DENSITY_MODEL = None
FOG_DENSITY_MODEL_KEY = None

### Run settings

Choose the profile and run settings here. Versions belong to each entry in `PROFILE_SETTINGS` above. The predictor checkpoint is also configured here; the **Load the fog-density model** step below actually opens it and freezes its weights.

In [ ]:
# Edit the profile version above and these run settings.
RUN_ENVIRONMENT = "colab"  # colab | local
TRAINING_PROFILE = PROFILE_2K
LOCAL_ROOT = Path.home() / DATA_FOLDER
ENVIRONMENT_PROFILES = {
    "colab": {
        "runtime_root": f"/content/{DATA_FOLDER}",
        "data_location": {"kind": "drive", "path": DATA_FOLDER},
        "output_location": {"kind": "drive", "path": f"{DATA_FOLDER}/runs"},
        "data_mode": "zip",
        "low_disk": True,
        "device": "cuda:0",
        "amp": True,
    },
    "local": {
        "runtime_root": str(LOCAL_ROOT / "runtime"),
        "data_location": {"kind": "path", "path": str(LOCAL_ROOT / "input")},
        "output_location": {"kind": "path", "path": str(LOCAL_ROOT / "runs")},
        "data_mode": "zip",
        "low_disk": False,
        "device": "cpu",
        "amp": False,
    },
}
if (
    RUN_ENVIRONMENT not in ENVIRONMENT_PROFILES
    or TRAINING_PROFILE not in PROFILE_SETTINGS
):
    raise ValueError("Choose a listed environment and training profile.")
ENVIRONMENT = deepcopy(ENVIRONMENT_PROFILES[RUN_ENVIRONMENT])
SELECTED_PROFILE = deepcopy(PROFILE_SETTINGS[TRAINING_PROFILE])
version = (
    SELECTED_PROFILE.get("version", "") if TRAINING_PROFILE != PROFILE_SETUP else ""
)
run_name = SELECTED_PROFILE["run_name"] + (f"-{version}" if version else "")
SELECTED_PROFILE.update(
    name=TRAINING_PROFILE,
    run_name=run_name,
    run_suffix=f"-{run_name}",
    initialization="existing_full_run"
    if TRAINING_PROFILE == PROFILE_FULL
    else "pretrained",
)
SETTINGS = {
    "retrain": False,
    "profile": SELECTED_PROFILE,
    "density_predictor": {
        "checkpoint": "",
        "source_run": FULL_RUN_NAME,
        "seed": DEFAULT_SEED,
    },
    "evaluation": {
        "source_run": "",
        "output_name": run_name + "-matched-evaluation",
        "cityscapes_root": str(
            Path(DRIVE_MOUNT) / DRIVE_SUBDIRECTORY / DATA_FOLDER / "cityscapes-labeled"
        ),
        "cityscapes_archive": str(
            Path(DRIVE_MOUNT) / DRIVE_SUBDIRECTORY / DATA_FOLDER / CITYSCAPES_ARCHIVE
        ),
        "split": "test",
        "max_pairs": 100,
    },
    "test": {
        "enabled": False,
        "models": [YOLO_SMALL],
        "seed": DEFAULT_SEED,
        "epochs": 1,
        "density_epochs": 1,
        "image_size": 320,
    },
    "runtime_root": ENVIRONMENT["runtime_root"],
    "drive_mount": DRIVE_MOUNT,
    "drive_subdir": DRIVE_SUBDIRECTORY,
    "data": {
        "location": ENVIRONMENT["data_location"],
        "mode": ENVIRONMENT["data_mode"],
        "archive": DATA_ARCHIVE,
        "url": DATA_ARCHIVE_URL,
        "low_disk": ENVIRONMENT["low_disk"],
        "minimum_free_gib": 15,
        "manifests": deepcopy(MANIFEST_PATHS),
        "city_images": "data/fog/cityscapes",
        "nuscenes": "data/depth/datasets/nuscenes",
        "allowed_archive_roots": ["data/", ".cache/"],
        "extraction_marker": ".data-extracted.json",
    },
    "output": {
        "location": ENVIRONMENT["output_location"],
        "run_name": run_name,
        "export_name": run_name + "-results",
        "packages_file": "colab-packages.txt",
    },
    "models": deepcopy(MODEL_SPECS),
    "training": {
        **deepcopy(TRAINING_DEFAULTS),
        "epochs": SELECTED_PROFILE["epochs"],
        "seeds": [SELECTED_PROFILE["seed"]]
        if TRAINING_PROFILE != PROFILE_FULL
        else list(FULL_RUN_SEEDS),
        "arms": list(DETECTION_ARMS),
        "size": SELECTED_PROFILE["size"],
        "save_every": SELECTED_PROFILE["save_every"],
        "amp": ENVIRONMENT["amp"],
        "device": ENVIRONMENT["device"],
    },
}
print(
    f"Profile: {TRAINING_PROFILE} | version: {version or 'base'} | output: {run_name}"
)
print(
    f"Device: {SETTINGS['training']['device']} | models: {', '.join(SELECTED_PROFILE['models'])}"
)

### Connect Google Drive before setup

Authorize Drive now, before dependency installation. Local filesystem profiles skip this step.


In [ ]:
# Authenticate before package installation or other long-running setup.
if any(
    SETTINGS[section]["location"]["kind"] == "drive" for section in ("data", "output")
):
    from google.colab import drive

    drive.mount(SETTINGS["drive_mount"])
    drive_root = Path(SETTINGS["drive_mount"]) / SETTINGS["drive_subdir"]
    if not drive_root.is_dir():
        raise RuntimeError("Drive is unavailable. Rerun this connection cell.")
    print("Google Drive connected. Continuing setup.")
else:
    print("Using local filesystem paths; no Drive connection needed.")

## 2. Set up the runtime

Choose storage, create the run folders, install dependencies, and check the GPU. All study implementation stays in this notebook.

In [ ]:
ROOT = Path(SETTINGS["runtime_root"])
if not ROOT.is_absolute():
    raise ValueError("runtime_root must be an absolute path.")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

### Install packages and check the GPU

In [ ]:
REQUIRED_PACKAGES = [
    "matplotlib",
    "numpy",
    "pandas",
    "pillow",
    "ijson",
    "shapely",
    "pyquaternion",
    "scipy",
    "filelock",
    "gdown",
    "tqdm",
    "ultralytics==8.4.155",
    "transformers==5.17.0",
    "pycocotools==2.0.11",
]
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--only-binary=:all:",
        *REQUIRED_PACKAGES,
    ]
)

### Supporting functions and classes (collapsed)

All implementation lives below, organized by purpose. These cells define the functions used by the visible steps; they do not run training or create Python files.

#### Detector models and detection scoring

#### Image manifests and deterministic data selection

In [ ]:
# fmt: off
#@title Supporting code: Image manifests and deterministic data selection (collapsed)
# fmt: on
import contextlib
import math
import gc
import hashlib
import importlib.metadata
import io
import json
import random
import shutil
import sqlite3
import stat
import zipfile
from collections import Counter
from dataclasses import dataclass, field  # noqa: F401
from pathlib import Path, PurePosixPath
from types import SimpleNamespace
from uuid import uuid4

import ijson
import numpy as np
import pandas as pd
import torch
from filelock import FileLock
from IPython.display import Image as PreviewImage, display
from PIL import Image
from pyquaternion import Quaternion
from shapely.geometry import MultiPoint, box as rectangle
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm


def set_reproducible_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def classify_density_bin(beta):
    """Map a known beta to a stable, interpretable fog-density stratum."""
    if beta is None:
        return "unknown"
    value = float(beta)
    for name, upper_bound in zip(DENSITY_BIN_NAMES, DENSITY_BIN_UPPER_BOUNDS):
        if value <= upper_bound:
            return name
    raise ValueError(f"Beta is outside the supported range: {value}")


def select_epoch_sample_indices(rows, seed, epoch, balanced=False):
    """Return a deterministic epoch order, optionally equalizing known beta strata."""
    if not rows:
        return []
    groups = {}
    for index, row in enumerate(rows):
        label = classify_density_bin(row.get("beta")) if balanced else "all"
        groups.setdefault(label, []).append(index)
    generator = random.Random(seed + epoch)
    if not balanced:
        return generator.sample(range(len(rows)), len(rows))
    target_size = max(len(indices) for indices in groups.values())
    order = []
    for label in sorted(groups):
        indices = groups[label]
        order.extend(generator.choices(indices, k=target_size))
    generator.shuffle(order)
    return order


def count_epoch_samples(rows, balanced=False):
    """Return the number of image steps represented by one configured epoch."""
    if not rows or not balanced:
        return len(rows)
    counts = {}
    for row in rows:
        label = classify_density_bin(row.get("beta"))
        counts[label] = counts.get(label, 0) + 1
    return max(counts.values()) * len(counts)


def sha256_digest(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream_json_records:
        for chunk in iter(lambda: stream_json_records.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def resolve_manifest_image(path, row, verify_images, seen_ids, seen_images):
    image = (path.parent / row["image"]).resolve()
    if row["id"] in seen_ids or str(image) in seen_images:
        raise ValueError("Duplicate image identity/path")
    if not image.is_file():
        raise FileNotFoundError(image)
    seen_ids.add(row["id"])
    seen_images.add(str(image))
    row["image"] = str(image)
    if verify_images or "width" not in row or "height" not in row:
        with Image.open(image) as source_image:
            width, height = source_image.size
            if verify_images:
                source_image.verify()
    else:
        width, height = row["width"], row["height"]
    if width <= 0 or height <= 0:
        raise ValueError("Invalid image dimensions")
    row["width"], row["height"] = width, height
    return width, height


def load_manifest(path, verify_images=False):
    """Load labeled images and check annotations and scene split boundaries."""
    path = Path(path)
    records = json.loads(path.read_text(encoding="utf-8"))
    if not records:
        raise ValueError("Manifest must contain labeled images.")
    seen_ids, seen_images, groups = set(), set(), {}
    for row in records:
        width, height = resolve_manifest_image(
            path, row, verify_images, seen_ids, seen_images
        )
        boxes = np.asarray(row["boxes"], dtype=float).reshape(-1, 4)
        labels = np.asarray(row["labels"])
        if len(boxes) != len(labels) or not np.isfinite(boxes).all():
            raise ValueError("Invalid boxes/labels")
        crowds = row.get("iscrowd", [0] * len(boxes))
        if len(crowds) != len(boxes) or any(flag not in (0, 1) for flag in crowds):
            raise ValueError("iscrowd must have one binary flag per annotation")
        if len(labels) and not np.isin(labels, range(len(CLASSES))).all():
            raise ValueError("Labels must use shared class IDs 0..6")
        if len(boxes) and (
            np.any(boxes[:, 2:] <= boxes[:, :2])
            or np.any(boxes < 0)
            or np.any(boxes > [width, height, width, height])
        ):
            raise ValueError("Boxes must be positive-area XYXY pixels inside the image")
        if row["dataset"] not in ("foggy_cityscapes", "nuscenes"):
            raise ValueError("Unknown dataset")
        if row["split"] not in ("density_train", "detector_train", "val", "test"):
            raise ValueError("Invalid split")
        if not row["id"] or not row["group"]:
            raise ValueError("Empty image/group identity")
        beta = row["beta"]
        if beta is not None and (not np.isfinite(beta) or not 0 <= beta <= BETA_SCALE):
            raise ValueError(
                "Beta must be null or an attenuation coefficient in [0, .02] m^-1"
            )
        if row["dataset"] == "foggy_cityscapes" and beta is None:
            raise ValueError("Foggy Cityscapes requires known beta")
        group_key = (row["dataset"], row["group"])
        if group_key in groups and groups[group_key] != row["split"]:
            raise ValueError("Scene/sequence leakage across splits")
        groups[group_key] = row["split"]
    return records


def select_complete_groups(rows, limit=None):
    """Select whole sequences, preserving all fog variants; None selects everything."""
    keys = sorted({(r["dataset"], r["group"]) for r in rows})
    if limit is not None:
        if limit < 1:
            raise ValueError("Group limit must be positive or None")
        keys = keys[:limit]
    keys = set(keys)
    return sorted(
        [r for r in rows if (r["dataset"], r["group"]) in keys], key=lambda r: r["id"]
    )


def deterministic_order(values, seed, label):
    return sorted(
        values,
        key=lambda value: hashlib.sha256(
            f"{PROFILE_SELECTION_SCHEMA}:{seed}:{label}:{value}".encode()
        ).hexdigest(),
    )


def select_scene_safe_rows(rows, image_cap, seed, split, label):
    """Select whole scenes by detector split without crossing validation boundaries."""
    candidates = [
        row for row in rows if row.get("detector_split", row["split"]) == split
    ]
    if image_cap is None:
        return sorted(candidates, key=lambda row: row["id"])
    if not isinstance(image_cap, int) or image_cap < 1:
        raise ValueError("Profile image caps must be positive integers or None")
    groups = {}
    for row in candidates:
        groups.setdefault((row["dataset"], row["group"]), []).append(row)
    selected, count = [], 0
    for group in deterministic_order(groups, seed, label):
        group_rows = sorted(groups[group], key=lambda row: row["id"])
        if count and count + len(group_rows) > image_cap:
            continue
        selected.extend(group_rows)
        count += len(group_rows)
        if count >= image_cap:
            break
    if not selected and candidates:
        selected = sorted(min(groups.values(), key=len), key=lambda row: row["id"])
    return sorted(selected, key=lambda row: row["id"])


def select_profile_training_rows(rows, profile, seed, training_image_cap=None):
    """Cap training rows evenly by dataset while keeping each scene intact."""
    if profile not in PROFILE_NAMES:
        raise ValueError(f"Unknown training profile: {profile}")
    training_rows = [
        row
        for row in rows
        if row.get("detector_split", row["split"]) == "detector_train"
    ]
    cap = training_image_cap
    if cap is None:
        cap = PROFILE_SETTINGS[profile]["training_image_cap"]
    datasets = sorted({row["dataset"] for row in training_rows})
    dataset_cap = None if cap is None else max(1, cap // max(len(datasets), 1))
    selected_ids = set()
    for dataset in datasets:
        dataset_rows = [row for row in training_rows if row["dataset"] == dataset]
        selected_ids.update(
            row["id"]
            for row in select_scene_safe_rows(
                dataset_rows,
                dataset_cap,
                seed,
                "detector_train",
                f"{profile}:{dataset}",
            )
        )
    return sorted(
        [
            row
            for row in rows
            if row.get("detector_split", row["split"]) != "detector_train"
            or row["id"] in selected_ids
        ],
        key=lambda row: row["id"],
    )

#### Detector models and detection metrics

In [ ]:
# fmt: off
#@title Supporting code: Detector models and detection metrics (collapsed)
# fmt: on
def load_image_tensor(row, size, device):
    with Image.open(row["image"]) as image:
        image = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
        array = np.asarray(image).copy()
    return (
        torch.from_numpy(array).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255
    )


class FogAdapter(nn.Module):
    """Identity-initialized RGB residual with scalar FiLM at hidden features.

    The control has exactly the same parameters, but always receives zero.
    This tests input adaptation, not internal YOLO/DETR feature conditioning.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Conv2d(3, 8, 3, padding=1)
        self.film = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16))
        self.output = nn.Conv2d(8, 3, 1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, image, beta):
        scale, shift = self.film(beta.reshape(-1, 1) / BETA_SCALE).chunk(2, dim=1)
        features = self.features(image)
        features = features * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        return (image + 0.1 * self.output(F.relu(features)).tanh()).clamp(0, 1)


def load_yolo_detection_model(checkpoint, device):
    # Import torchvision first so Ultralytics uses its compiled NMS implementation.
    import torchvision.ops  # noqa: F401
    from ultralytics import YOLO
    from ultralytics.utils import DEFAULT_CFG_DICT

    model = YOLO(str(checkpoint)).model.to(device).float()
    model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
    return model


def load_detr_detection_model(checkpoint, device):
    from transformers import DetrConfig, DetrForObjectDetection

    options = {} if Path(checkpoint).is_dir() else {"revision": "no_timm"}
    raw_config, _ = DetrConfig.get_config_dict(str(checkpoint), **options)
    if raw_config.get("dilation") is None:
        raw_config["dilation"] = False
    if raw_config.get("use_pretrained_backbone") is None:
        raw_config["use_pretrained_backbone"] = False
    model_config = DetrConfig.from_dict(raw_config)
    return DetrForObjectDetection.from_pretrained(
        str(checkpoint), config=model_config, **options
    ).to(device)


def freeze_yolo_distribution_integral(model):
    for parameter_name, parameter in model.named_parameters():
        if ".dfl." in parameter_name:
            parameter.requires_grad_(False)


def load_detection_model(name, checkpoint, device, trainable):
    is_yolo = name in (YOLO_SMALL, YOLO_MEDIUM)
    model = (
        load_yolo_detection_model(checkpoint, device)
        if is_yolo
        else load_detr_detection_model(checkpoint, device)
    )
    model.eval().requires_grad_(trainable)
    if is_yolo:
        freeze_yolo_distribution_integral(model)
    return model, is_yolo


def format_detection_targets(images, annotation, class_mapping, device):
    boxes = images.new_tensor(annotation["boxes"]).reshape(-1, 4)
    boxes = boxes / images.new_tensor([annotation["width"], annotation["height"]] * 2)
    center = (boxes[:, :2] + boxes[:, 2:]) / 2
    size = boxes[:, 2:] - boxes[:, :2]
    boxes = torch.cat([center, size], dim=1)
    labels = torch.tensor(
        [class_mapping[index] for index in annotation["labels"]], device=device
    )
    return boxes, labels


def normalize_detr_pixels(images):
    mean = images.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
    standard_deviation = images.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
    return (images - mean) / standard_deviation


def calculate_yolo_training_loss(model, images, boxes, labels):
    batch = {
        "img": images,
        "batch_idx": images.new_zeros(len(labels)),
        "cls": labels.reshape(-1, 1),
        "bboxes": boxes,
    }
    return model.loss(batch, model(images))[0].sum()


def calculate_detr_training_loss(model, images, boxes, labels):
    return model(
        pixel_values=normalize_detr_pixels(images),
        labels=[{"class_labels": labels.long(), "boxes": boxes}],
    ).loss


def map_normalized_boxes_to_image(boxes, images, annotation):
    pixel_bounds = boxes.clamp(0, 1) * images.new_tensor(
        [annotation["width"], annotation["height"]] * 2
    )
    return pixel_bounds.detach().cpu().tolist()


def convert_model_detections_to_rows(
    boxes, scores, labels, class_mapping, images, annotation
):
    rows = []
    pixel_boxes = map_normalized_boxes_to_image(boxes, images, annotation)
    for box, score, label in zip(
        pixel_boxes,
        scores.detach().cpu().tolist(),
        labels.detach().cpu().tolist(),
    ):
        if (
            label in class_mapping
            and score >= 0.001
            and box[2] > box[0]
            and box[3] > box[1]
        ):
            rows.append(
                {"box": box, "score": score, "label": class_mapping.index(label)}
            )
    return sorted(rows, key=lambda row: -row["score"])[:100]


def predict_yolo_objects(model, images, annotation):
    from ultralytics.utils.nms import non_max_suppression

    output = model(images)
    result = non_max_suppression(
        output, conf_thres=0.001, iou_thres=0.7, classes=YOLO_IDS, max_det=100
    )[0]
    boxes, scores, labels = result[:, :4], result[:, 4], result[:, 5].long()
    boxes = boxes / images.shape[-1]
    return convert_model_detections_to_rows(
        boxes, scores, labels, YOLO_IDS, images, annotation
    )


def predict_detr_objects(model, images, annotation):
    output = model(pixel_values=normalize_detr_pixels(images))
    probabilities = output.logits.softmax(-1)[0, :, :-1]
    scores, labels = probabilities.max(-1)
    center, size = output.pred_boxes[0].split(2, dim=-1)
    boxes = torch.cat([center - size / 2, center + size / 2], dim=-1)
    return convert_model_detections_to_rows(
        boxes, scores, labels, COCO_IDS, images, annotation
    )


class ObjectDetector:
    """Expose one training and prediction interface for YOLO and DETR."""

    def __init__(self, name, checkpoint, device, trainable=True):
        self.name, self.device = name, device
        self.is_yolo = name in (YOLO_SMALL, YOLO_MEDIUM)
        self.model, self.is_yolo = load_detection_model(
            name, checkpoint, device, trainable
        )

    def set_training_mode(self):
        self.model.train()
        # Batch-one microsteps preserve pretrained batch-normalization statistics.
        for module in self.model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    def forward(self, images):
        if self.is_yolo:
            return self.model(images)
        return self.model(pixel_values=normalize_detr_pixels(images))

    def training_loss(self, images, annotation):
        mapping = YOLO_IDS if self.is_yolo else COCO_IDS
        boxes, labels = format_detection_targets(
            images, annotation, mapping, self.device
        )
        if self.is_yolo:
            return calculate_yolo_training_loss(self.model, images, boxes, labels)
        return calculate_detr_training_loss(self.model, images, boxes, labels)

    def predict_objects(self, images, annotation):
        if self.is_yolo:
            return predict_yolo_objects(self.model, images, annotation)
        return predict_detr_objects(self.model, images, annotation)


def get_density_feature(arm, x, row, density):
    if arm == ARM_RGB:
        return x.new_zeros((1, 1))
    if arm == ARM_DENSITY:
        return get_fog_density(x, density)
    if arm == ARM_ORACLE and row["beta"] is not None:
        return x.new_tensor([[row["beta"]]])
    raise ValueError(
        "Oracle density is unavailable; do not substitute zero or a prediction"
    )


def build_coco_dataset(rows, results):
    if [row["id"] for row in rows] != [result["id"] for result in results]:
        raise ValueError("Predictions are not paired with the exact evaluation images")
    images, annotations, predictions = [], [], []
    for image_index, (row, result) in enumerate(zip(rows, results), 1):
        images.append(
            {"id": image_index, "width": row["width"], "height": row["height"]}
        )
        crowds = row.get("iscrowd", [0] * len(row["boxes"]))
        for box, label, crowd in zip(row["boxes"], row["labels"], crowds):
            x1, y1, x2, y2 = box
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": image_index,
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": int(crowd),
                }
            )
        for prediction in result["prediction"]:
            x1, y1, x2, y2 = prediction["box"]
            predictions.append(
                {
                    "image_id": image_index,
                    "category_id": prediction["label"],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": prediction["score"],
                }
            )
    return images, annotations, predictions


def evaluate_coco_dataset(images, annotations, predictions):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    truth = COCO()
    truth.dataset = {
        "info": {},
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": index, "name": name} for index, name in enumerate(CLASSES)
        ],
    }
    with contextlib.redirect_stdout(io.StringIO()):
        truth.createIndex()
        if predictions:
            predicted = truth.loadRes(predictions)
        else:
            predicted = COCO()
            predicted.dataset = {**truth.dataset, "annotations": []}
            predicted.createIndex()
        evaluator = COCOeval(truth, predicted, "bbox")
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator


def calculate_coco_summary_metrics(evaluator):
    true_positives = false_positives = positive_annotations = 0
    for image_result in evaluator.evalImgs:
        if image_result is None or image_result["aRng"] != evaluator.params.areaRng[0]:
            continue
        keep = np.asarray(image_result["dtScores"]) >= 0.25
        ignored = np.asarray(image_result["dtIgnore"])[0].astype(bool)
        matched = np.asarray(image_result["dtMatches"])[0] > 0
        true_positives += int(np.sum(keep & ~ignored & matched))
        false_positives += int(np.sum(keep & ~ignored & ~matched))
        positive_annotations += int(
            np.sum(~np.asarray(image_result["gtIgnore"]).astype(bool))
        )
    metrics = {
        "mAP50_95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AR100": float(evaluator.stats[8]),
        "precision_at_025": (
            true_positives / (true_positives + false_positives)
            if true_positives + false_positives
            else 0.0
        ),
        "recall_at_025": (
            true_positives / positive_annotations if positive_annotations else np.nan
        ),
    }
    for class_index, class_name in enumerate(CLASSES):
        values = evaluator.eval["precision"][:, :, class_index, 0, -1]
        supported = values[values >= 0]
        metrics[f"AP_{class_name}"] = (
            float(supported.mean()) if len(supported) else np.nan
        )
    return {key: np.nan if value == -1 else value for key, value in metrics.items()}


def calculate_coco_precision_recall_curve(evaluator):
    precision = evaluator.eval["precision"][0, :, :, 0, -1]
    valid = precision >= 0
    counts = valid.sum(axis=1)
    average_precision = np.divide(
        np.where(valid, precision, 0).sum(axis=1),
        counts,
        out=np.full(len(counts), np.nan),
        where=counts > 0,
    )
    return average_precision.tolist(), evaluator.params.recThrs.tolist()


def calculate_coco_detection_metrics(rows, results, include_curve=False):
    """Calculate COCO detection metrics for paired labeled images."""
    images, annotations, predictions = build_coco_dataset(rows, results)
    evaluator = evaluate_coco_dataset(images, annotations, predictions)
    metrics = calculate_coco_summary_metrics(evaluator)
    if include_curve:
        metrics["precision_curve"], metrics["recall_curve"] = (
            calculate_coco_precision_recall_curve(evaluator)
        )
    return metrics


def density_summary_groups(rows):
    groups = [("all", "all", list(range(len(rows))))]
    foggy_indices = [
        index
        for index, row in enumerate(rows)
        if row["beta"] is not None and row["beta"] > 0
    ]
    if foggy_indices:
        groups.append(("foggy", "foggy", foggy_indices))
    for beta in sorted({row["beta"] for row in rows if row["beta"] is not None}):
        groups.append(
            (
                str(beta),
                classify_density_bin(beta),
                [index for index, row in enumerate(rows) if row["beta"] == beta],
            )
        )
    unknown_indices = [index for index, row in enumerate(rows) if row["beta"] is None]
    if unknown_indices:
        groups.append(("unknown", "unknown", unknown_indices))
    return groups


def summarize_density_group(rows, results, bucket, bin_name, indices):
    selected_rows = [rows[index] for index in indices]
    selected_results = [results[index] for index in indices]
    image_timings = [
        timing for result in selected_results for timing in result["timings"]
    ]
    metrics = calculate_coco_detection_metrics(selected_rows, selected_results)
    condition = (
        "clear"
        if bin_name == "clear"
        else "foggy"
        if bin_name in ("light", "moderate", "dense", "foggy")
        else "unknown"
        if bin_name == "unknown"
        else "mixed"
    )
    mean_latency = float(np.mean([timing["total_ms"] for timing in image_timings]))
    return {
        "density": bucket,
        "density_bin": bin_name,
        "condition": condition,
        "distribution": "natural",
        "images": len(selected_rows),
        **metrics,
        "mean_ms": mean_latency,
        "p50_ms": float(np.median([timing["total_ms"] for timing in image_timings])),
        "p95_ms": float(
            np.percentile([timing["total_ms"] for timing in image_timings], 95)
        ),
        "serial_fps": 1000 / mean_latency,
        "density_mean_ms": float(
            np.mean([timing["density_ms"] for timing in image_timings])
        ),
    }


def summarize_detection_metrics_by_density(rows, results):
    return [
        summarize_density_group(rows, results, bucket, bin_name, indices)
        for bucket, bin_name, indices in density_summary_groups(rows)
    ]

#### nuScenes data preparation

In [ ]:
# fmt: off
#@title Supporting code: nuScenes data preparation
# fmt: on
"""Resumable nuScenes preparation without loading millions of sweep records into RAM."""


CAMERAS = (
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_FRONT_LEFT",
)
SCHEMA = 3


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def stream_json_records(path):
    with Path(path).open("rb") as handle:
        yield from ijson.items(handle, "item", use_float=True)


def source_signature(root, version):
    names = (
        "scene",
        "sample",
        "sample_data",
        "sample_annotation",
        "ego_pose",
        "calibrated_sensor",
        "sensor",
        "instance",
        "category",
    )
    return {
        "schema": SCHEMA,
        "root": str(root),
        "version": version,
        "tables": {
            name: [
                ((root / version) / f"{name}.json").stat().st_size,
                ((root / version) / f"{name}.json").stat().st_mtime_ns,
            ]
            for name in names
        },
    }


def metadata_index_is_current(database_path, stamp_path, signature):
    return (
        database_path.is_file()
        and stamp_path.is_file()
        and json.loads(stamp_path.read_text()) == signature
    )


def create_metadata_index_database(connection):
    connection.execute("PRAGMA journal_mode=OFF")
    connection.execute("PRAGMA synchronous=OFF")
    connection.execute(
        "CREATE TABLE records(kind TEXT, token TEXT, parent TEXT, payload TEXT, PRIMARY KEY(kind,token))"
    )


def keep_nuscenes_metadata_row(kind, row, camera_poses):
    if kind == "sample_data":
        is_camera_keyframe = row["is_key_frame"] and row["filename"].startswith(
            "samples/CAM_"
        )
        if not is_camera_keyframe:
            return False
        camera_poses.add(row["ego_pose_token"])
    if kind == "ego_pose" and row["token"] not in camera_poses:
        return False
    return True


def import_nuscenes_metadata_table(connection, root, version, kind, camera_poses):
    batch = []
    for row in tqdm(
        stream_json_records(root / version / f"{kind}.json"),
        desc=f"Index {kind}",
        unit=" rows",
        mininterval=2,
    ):
        if not keep_nuscenes_metadata_row(kind, row, camera_poses):
            continue
        parent = row.get("sample_token", row.get("scene_token", ""))
        batch.append((kind, row["token"], parent, json.dumps(row)))
        if len(batch) >= 5000:
            connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
            batch.clear()
    if batch:
        connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
    connection.commit()


def build_nuscenes_metadata_index(root, version, cache):
    """Build or reuse the SQLite index used to prepare scene annotations."""
    signature = source_signature(root, version)
    database_path = cache / "metadata.sqlite"
    stamp_path = cache / "metadata-source.json"
    if metadata_index_is_current(database_path, stamp_path, signature):
        return database_path
    temporary_path = cache / "metadata-building.sqlite"
    if temporary_path.exists():
        temporary_path.unlink()
    connection = sqlite3.connect(temporary_path)
    try:
        create_metadata_index_database(connection)
        camera_poses = set()
        for kind in signature["tables"]:
            import_nuscenes_metadata_table(
                connection, root, version, kind, camera_poses
            )
        connection.execute("CREATE INDEX by_parent ON records(kind,parent)")
        connection.commit()
    finally:
        connection.close()
    os.replace(temporary_path, database_path)
    atomic_json(stamp_path, signature)
    return database_path


def clipped_bounds(points, width, height):
    """Clip the projected convex hull, not its bounding rectangle (not equivalent)."""
    if len(points) < 3 or not np.isfinite(points).all():
        return None
    lo, hi = points.min(axis=0), points.max(axis=0)
    if hi[0] <= 0 or hi[1] <= 0 or lo[0] >= width or lo[1] >= height:
        return None
    hull = MultiPoint(points).convex_hull.intersection(rectangle(0, 0, width, height))
    return list(hull.bounds) if not hull.is_empty and hull.area > 0 else None


def load_nuscenes_lookup_tables(connection):
    return {
        kind: {
            record["token"]: record for record in read_nuscenes_table(connection, kind)
        }
        for kind in ("sensor", "calibrated_sensor", "category", "instance", "scene")
    }


def read_nuscenes_table(connection, kind):
    return [
        json.loads(payload)
        for (payload,) in connection.execute(
            "SELECT payload FROM records WHERE kind=?", (kind,)
        )
    ]


def read_nuscenes_children(connection, kind, parent_token):
    return [
        json.loads(payload)
        for (payload,) in connection.execute(
            "SELECT payload FROM records WHERE kind=? AND parent=?",
            (kind, parent_token),
        )
    ]


def read_nuscenes_record(connection, kind, token):
    row = connection.execute(
        "SELECT payload FROM records WHERE kind=? AND token=?", (kind, token)
    ).fetchone()
    if row is None:
        raise KeyError(f"Missing nuScenes {kind} record: {token}")
    return json.loads(row[0])


def select_nuscenes_scenes(connection, version):
    from nuscenes.utils.splits import create_splits_scenes

    scenes = {
        scene["token"]: scene for scene in read_nuscenes_table(connection, "scene")
    }
    official_splits = create_splits_scenes()
    train_key = "mini_train" if version == "v1.0-mini" else "train"
    test_key = "mini_val" if version == "v1.0-mini" else "val"
    train_names = set(official_splits[train_key])
    test_names = set(official_splits[test_key])
    selected = sorted(
        (
            scene
            for scene in scenes.values()
            if scene["name"] in train_names | test_names
        ),
        key=lambda scene: scene["name"],
    )
    return selected, train_names, test_names


def create_nuscenes_coverage_report(root, version, expected_scenes, selected_scenes):
    return {
        "source": source_signature(root, version),
        "expected_scenes": expected_scenes,
        "selected_scenes": len(selected_scenes),
        "expected_images": 0,
        "available_images": 0,
        "missing_images": [],
        "missing_channels": [],
    }


def choose_nuscenes_scene_split(scene_name, train_names, test_names):
    if scene_name in test_names:
        return "test"
    bucket = int(hashlib.sha256(scene_name.encode()).hexdigest()[:8], 16) % 10
    return "val" if bucket == 0 else "detector_train"


def scene_shard_is_reusable(shard_path, source_signature_value):
    if not shard_path.is_file():
        return False
    data = json.loads(shard_path.read_text(encoding="utf-8"))
    return (
        data.get("source") == source_signature_value
        and not data["missing_images"]
        and all(Path(row["image"]).is_file() for row in data["rows"])
    )


def collect_nuscenes_world_boxes(sample, connection, lookup_tables, box_type):
    world_boxes = []
    for annotation in read_nuscenes_children(
        connection, "sample_annotation", sample["token"]
    ):
        instance = lookup_tables["instance"][annotation["instance_token"]]
        category = lookup_tables["category"][instance["category_token"]]["name"]
        class_name = nuscenes_detection_class(category)
        if class_name is None:
            continue
        box = box_type(
            annotation["translation"],
            annotation["size"],
            Quaternion(annotation["rotation"]),
        )
        world_boxes.append((box.corners(), CLASSES.index(class_name)))
    return world_boxes


def nuscenes_detection_class(category):
    if category.startswith("human.pedestrian.") and category.rsplit(".", 1)[-1] not in (
        "stroller",
        "wheelchair",
        "personal_mobility",
    ):
        return "person"
    if category.startswith("vehicle.bus."):
        return "bus"
    return {
        "vehicle.car": "car",
        "vehicle.truck": "truck",
        "vehicle.bicycle": "bicycle",
        "vehicle.motorcycle": "motorcycle",
    }.get(category)


def project_nuscenes_box_to_camera(corners, calibrated_sensor, ego_pose, width, height):
    ego_rotation = Quaternion(ego_pose["rotation"]).rotation_matrix.T
    camera_rotation = Quaternion(calibrated_sensor["rotation"]).rotation_matrix.T
    rotation = camera_rotation @ ego_rotation
    translation = -rotation @ np.array(
        ego_pose["translation"]
    ) - camera_rotation @ np.array(calibrated_sensor["translation"])
    camera_points = rotation @ corners + translation[:, None]
    camera_points = camera_points[:, camera_points[2] > 0]
    if camera_points.shape[1] < 3:
        return None
    intrinsic = np.array(calibrated_sensor["camera_intrinsic"])
    projected = intrinsic @ camera_points
    image_points = (projected[:2] / projected[2]).T
    return clipped_bounds(image_points, width, height)


def build_nuscenes_camera_row(
    sample_data, scene, split, root, connection, lookup_tables, world_boxes
):
    calibrated = lookup_tables["calibrated_sensor"][
        sample_data["calibrated_sensor_token"]
    ]
    sensor = lookup_tables["sensor"][calibrated["sensor_token"]]
    channel = sensor["channel"]
    image_path = root / sample_data["filename"]
    if not image_path.is_file():
        return None, channel
    ego_pose = read_nuscenes_record(
        connection, "ego_pose", sample_data["ego_pose_token"]
    )
    boxes, labels = [], []
    for corners, class_id in world_boxes:
        bounds = project_nuscenes_box_to_camera(
            corners, calibrated, ego_pose, sample_data["width"], sample_data["height"]
        )
        if bounds is not None:
            boxes.append(bounds)
            labels.append(class_id)
    return (
        {
            "id": f"nuscenes/{sample_data['token']}",
            "image": str(image_path),
            "dataset": "nuscenes",
            "group": scene["name"],
            "split": split,
            "beta": None,
            "camera": channel,
            "width": sample_data["width"],
            "height": sample_data["height"],
            "boxes": boxes,
            "labels": labels,
            "iscrowd": [0] * len(boxes),
            "label_policy": "projected 3D boxes, all visibility levels; 2D proxy benchmark",
        },
        channel,
    )


def project_nuscenes_sample(
    sample, scene, split, root, connection, lookup_tables, box_type
):
    world_boxes = collect_nuscenes_world_boxes(
        sample, connection, lookup_tables, box_type
    )
    rows, missing_images, available_channels = [], [], set()
    for sample_data in read_nuscenes_children(
        connection, "sample_data", sample["token"]
    ):
        calibrated = lookup_tables["calibrated_sensor"][
            sample_data["calibrated_sensor_token"]
        ]
        channel = lookup_tables["sensor"][calibrated["sensor_token"]]["channel"]
        if channel not in CAMERAS:
            continue
        available_channels.add(channel)
        row, _ = build_nuscenes_camera_row(
            sample_data, scene, split, root, connection, lookup_tables, world_boxes
        )
        if row is None:
            missing_images.append(str(root / sample_data["filename"]))
        else:
            rows.append(row)
    missing_channels = [
        f"{sample['token']}/{channel}" for channel in set(CAMERAS) - available_channels
    ]
    return rows, missing_images, missing_channels


def create_nuscenes_scene_shard(
    scene,
    train_names,
    test_names,
    source,
    root,
    connection,
    lookup_tables,
    box_type,
    shard_path,
):
    split = choose_nuscenes_scene_split(scene["name"], train_names, test_names)
    scene_data = {
        "rows": [],
        "expected_images": 0,
        "missing_images": [],
        "missing_channels": [],
        "source": source,
    }
    samples = sorted(
        read_nuscenes_children(connection, "sample", scene["token"]),
        key=lambda sample: sample["timestamp"],
    )
    for sample in samples:
        rows, missing_images, missing_channels = project_nuscenes_sample(
            sample, scene, split, root, connection, lookup_tables, box_type
        )
        scene_data["rows"].extend(rows)
        scene_data["expected_images"] += len(CAMERAS)
        scene_data["missing_images"].extend(missing_images)
        scene_data["missing_channels"].extend(missing_channels)
    atomic_json(shard_path, scene_data)
    return scene_data


def load_or_create_nuscenes_scene_shard(
    scene,
    cache,
    report,
    train_names,
    test_names,
    root,
    connection,
    lookup_tables,
    box_type,
):
    shard_path = cache / f"{scene['name']}.json"
    if scene_shard_is_reusable(shard_path, report["source"]):
        return json.loads(shard_path.read_text(encoding="utf-8")), shard_path
    data = create_nuscenes_scene_shard(
        scene,
        train_names,
        test_names,
        report["source"],
        root,
        connection,
        lookup_tables,
        box_type,
        shard_path,
    )
    return data, shard_path


def add_scene_shard_to_coverage(report, scene_data):
    report["expected_images"] += scene_data["expected_images"]
    report["available_images"] += len(scene_data["rows"])
    report["missing_images"].extend(scene_data["missing_images"])
    report["missing_channels"].extend(scene_data["missing_channels"])


def write_nuscenes_manifest(shard_paths, output):
    temporary = output.with_suffix(".json.tmp")
    counts = Counter()
    with temporary.open("w", encoding="utf-8") as handle:
        handle.write("[")
        first_row = True
        for shard_path in shard_paths:
            for row in json.loads(shard_path.read_text(encoding="utf-8"))["rows"]:
                if not first_row:
                    handle.write(",\n")
                handle.write(json.dumps(row))
                first_row = False
                counts[row["split"]] += 1
        handle.write("]")
    os.replace(temporary, output)
    return counts


def prepare_nuscenes_scene_shards(
    scenes,
    train_names,
    test_names,
    root,
    version,
    output,
    cache,
    connection,
    lookup_tables,
    box_type,
):
    report = create_nuscenes_coverage_report(
        root, version, len(train_names | test_names), scenes
    )
    shard_paths = []
    for scene in tqdm(scenes, desc="Project scenes (resumable)", mininterval=2):
        scene_data, shard_path = load_or_create_nuscenes_scene_shard(
            scene,
            cache,
            report,
            train_names,
            test_names,
            root,
            connection,
            lookup_tables,
            box_type,
        )
        add_scene_shard_to_coverage(report, scene_data)
        shard_paths.append(shard_path)
        if len(shard_paths) % 25 == 0:
            atomic_json(output.with_suffix(".coverage.json"), report)
    return report, shard_paths


def prepare_nuscenes_manifest(root, output, version="v1.0-trainval"):
    """Create and verify the resumable nuScenes scene manifest."""
    from nuscenes.utils.data_classes import Box

    root, output = Path(root).resolve(), Path(output).resolve()
    cache = output.parent / (output.stem + "-preparation")
    cache.mkdir(parents=True, exist_ok=True)
    with FileLock(str(cache / "prepare.lock"), timeout=0):
        database = build_nuscenes_metadata_index(root, version, cache)
        connection = sqlite3.connect(database)
        connection.execute("PRAGMA cache_size=-64000")
        try:
            lookup_tables = load_nuscenes_lookup_tables(connection)
            scenes, train_names, test_names = select_nuscenes_scenes(
                connection, version
            )
            report, shard_paths = prepare_nuscenes_scene_shards(
                scenes,
                train_names,
                test_names,
                root,
                version,
                output,
                cache,
                connection,
                lookup_tables,
                Box,
            )
        finally:
            connection.close()
    complete = (
        len(scenes) == len(train_names | test_names)
        and not report["missing_images"]
        and not report["missing_channels"]
    )
    report["complete"] = complete
    atomic_json(output.with_suffix(".coverage.json"), report)
    if not complete:
        raise RuntimeError(
            f"nuScenes coverage incomplete: {report['available_images']}/{report['expected_images']} images. "
            f"See {output.with_suffix('.coverage.json')}. Download missing images, then rerun."
        )
    counts = write_nuscenes_manifest(shard_paths, output)
    print(
        f"Prepared {report['available_images']} images; complete=True; splits={dict(counts)}",
        flush=True,
    )
    return output

#### Image manifests and data validation

In [ ]:
# fmt: off
#@title Supporting code: Image manifests and data validation
# fmt: on
"""Dataset entry points; nuScenes preparation lives in fog_prepare."""


def resolve_foggy_images(project_root, explicit=None, download=False):
    """Reuse Windows/Linux Kaggle caches from the user's downloader; optional same-source download."""
    import os

    project_root = Path(project_root).resolve()
    candidates = [Path(explicit)] if explicit else []
    candidates += [
        project_root / "data/fog/cityscapes",
        Path.home() / "Downloads" / "foggy-cityscapes",
        Path.home() / "Downloads" / "Foggy Cityscapes",
        project_root
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
        Path.home()
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
    ]
    for path in candidates:
        if path.is_dir() and (
            (path / "leftImg8bit_foggy").is_dir()
            or (path / "train").is_dir()
            or next(path.rglob("leftImg8bit_foggy"), None) is not None
        ):
            return path
    if not download:
        return None
    # Keep downloads inside this workspace. Version 1 matches Untitled2.ipynb.
    os.environ["KAGGLEHUB_CACHE"] = str(project_root / ".cache/kagglehub")
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError(
            "Install requirements-fog-research.txt to enable the optional Kaggle download"
        ) from error
    print(
        "Downloading the same Foggy Cityscapes v1 source as Untitled2.ipynb (~29.1 GB archive plus extraction).",
        flush=True,
    )
    return Path(
        kagglehub.dataset_download("minhducnguyen9705/foggy-cityscapes/versions/1")
    )


def find_foggy_cityscapes_image_root(root):
    direct_root = root / "leftImg8bit_foggy"
    if direct_root.is_dir():
        candidates = [direct_root]
    elif (root / "train").is_dir():
        candidates = [root]
    else:
        candidates = list(root.rglob("leftImg8bit_foggy"))
    if len(candidates) != 1:
        raise ValueError(
            f"Expected one Foggy Cityscapes image root under {root}; found {len(candidates)}"
        )
    return candidates[0]


def extract_cityscapes_instances(mask_path, with_detection_labels):
    if not with_detection_labels:
        return [], [], []
    city_ids = {
        24: "person",
        25: "person",
        26: "car",
        27: "truck",
        28: "bus",
        31: "train",
        32: "motorcycle",
        33: "bicycle",
    }
    if not mask_path.is_file():
        raise FileNotFoundError(f"Missing instance labels: {mask_path}")
    with Image.open(mask_path) as mask_image:
        mask = np.array(mask_image)
    boxes, labels, crowds = [], [], []
    for instance in np.unique(mask):
        category = int(instance // 1000 if instance >= 1000 else instance)
        if category not in city_ids:
            continue
        y_coordinates, x_coordinates = np.where(mask == instance)
        boxes.append(
            [
                int(x_coordinates.min()),
                int(y_coordinates.min()),
                int(x_coordinates.max()) + 1,
                int(y_coordinates.max()) + 1,
            ]
        )
        labels.append(CLASSES.index(city_ids[category]))
        crowds.append(int(instance < 1000))
    return boxes, labels, crowds


def cityscapes_group_split(group, split, with_detection_labels):
    if split == "val":
        return "test"
    bucket = int(hashlib.sha256(group.encode()).hexdigest()[:8], 16) % 10
    if with_detection_labels:
        if bucket < 2:
            return "density_train"
        return "val" if bucket < 4 else "detector_train"
    return "val" if bucket < 2 else "density_train"


def build_foggy_cityscapes_row(root, image, split, with_detection_labels, cache):
    base, suffix = image.name.split("_leftImg8bit_foggy_beta_")
    beta = float(suffix.removesuffix(".png"))
    mask_path = (
        root / "gtFine" / split / image.parent.name / f"{base}_gtFine_instanceIds.png"
    )
    if base not in cache:
        with Image.open(image) as source_image:
            width, height = source_image.size
        boxes, labels, crowds = extract_cityscapes_instances(
            mask_path, with_detection_labels
        )
        cache[base] = boxes, labels, crowds, width, height
    boxes, labels, crowds, width, height = cache[base]
    group = "_".join(base.split("_")[:2])
    assigned_split = cityscapes_group_split(group, split, with_detection_labels)
    return {
        "id": f"city/{base}/{beta}",
        "image": str(image),
        "dataset": "foggy_cityscapes",
        "group": group,
        "pair_id": f"city/{base}",
        "condition": "foggy",
        "split": assigned_split,
        "beta": beta,
        "boxes": boxes,
        "labels": labels,
        "iscrowd": crowds,
        "width": width,
        "height": height,
        "annotation_source": str(mask_path) if with_detection_labels else None,
        "detection_labeled": with_detection_labels,
        "label_policy": "instance-mask bounding rectangles; rider merged into person"
        if with_detection_labels
        else "beta-only; not a detection target",
    }



def prepare_cityscapes_data():
    """Extract the uploaded Cityscapes images and gtFine masks once."""
    import zipfile

    options = SETTINGS["evaluation"]
    archive_path = Path(options["cityscapes_archive"])
    root = Path(options["cityscapes_root"])
    marker = root / ".cityscapes-extracted"
    if marker.exists():
        print(f"Cityscapes data is ready: {root}")
        return root
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Upload {CITYSCAPES_ARCHIVE} to MyDrive/{DATA_FOLDER}/ before continuing: {archive_path}"
        )

    archive_roots = (
        ("Cityscape Dataset/leftImg8bit/", root / "leftImg8bit"),
        ("Fine Annotations/gtFine/", root / "gtFine"),
    )
    root.mkdir(parents=True, exist_ok=True)
    extracted = 0
    with zipfile.ZipFile(archive_path) as cityscape_zip:
        for member in cityscape_zip.infolist():
            destination_root = next(
                (target for prefix, target in archive_roots if member.filename.startswith(prefix)),
                None,
            )
            if destination_root is None or member.is_dir():
                continue
            prefix = next(prefix for prefix, target in archive_roots if target == destination_root)
            relative_path = Path(member.filename[len(prefix):])
            destination = destination_root / relative_path
            destination.parent.mkdir(parents=True, exist_ok=True)
            with cityscape_zip.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            extracted += 1
            if extracted % 1000 == 0:
                print(f"Extracted {extracted:,} Cityscapes files...", flush=True)

    required_mask = (
        root / "gtFine/train/aachen/aachen_000000_000019_gtFine_instanceIds.png"
    )
    if not required_mask.is_file():
        raise FileNotFoundError(
            f"The Cityscapes archive did not contain the expected instance mask: {required_mask}"
        )
    marker.write_text(f"archive={archive_path.name}\nfiles={extracted}\n")
    print(f"Prepared {extracted:,} Cityscapes images and masks under {root}")
    return root


def load_labeled_cityscapes_data(fog_density_rows=None):
    """Load cached Foggy Cityscapes boxes and their matched clear images."""
    root = Path(SETTINGS["evaluation"]["cityscapes_root"])
    output = Path(TRAINING_CONFIGURATION["output"]) / LABELED_CITYSCAPES_FILE
    if fog_density_rows is None:
        fog_density_rows = load_study_manifest(TRAINING_CONFIGURATION["city_manifest"])
    fog_density_rows = [row for row in fog_density_rows if row["beta"] > 0]
    output.parent.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output) + ".lock", timeout=0):
        if not output.exists():
            split_by_group = {row["group"]: row["split"] for row in fog_density_rows}
            cache, rows = {}, []
            for fog_row in fog_density_rows:
                image = Path(fog_row["image"])
                source_split = image.parents[1].name
                row = build_foggy_cityscapes_row(root, image, source_split, True, cache)
                row["split"] = split_by_group[row["group"]]
                rows.append(row)
            add_matched_clear_cityscapes_rows(rows, root, True)
            for row in rows:
                row["detector_split"] = (
                    "detector_train"
                    if row["split"] == "density_train"
                    else row["split"]
                )
            atomic_json(output, rows)
    return load_manifest(output, verify_images=True)


def create_cityscapes_split_rows(image_root, root, split, with_detection_labels):
    from tqdm.auto import tqdm

    image_paths = sorted((image_root / split).glob("*/*_leftImg8bit_foggy_beta_*.png"))
    annotation_cache = {}
    rows = []
    for image in tqdm(image_paths, desc=f"Foggy Cityscapes {split}", mininterval=2):
        rows.append(
            build_foggy_cityscapes_row(
                root, image, split, with_detection_labels, annotation_cache
            )
        )
    return rows


def add_matched_clear_cityscapes_rows(rows, root, with_detection_labels):
    seen_bases = set()
    for foggy_row in list(rows):
        base = foggy_row["id"].split("/")[1]
        split = "val" if foggy_row["split"] == "test" else "train"
        clear_image = (
            root
            / "leftImg8bit"
            / split
            / base.split("_")[0]
            / f"{base}_leftImg8bit.png"
        )
        if with_detection_labels and not clear_image.is_file():
            raise FileNotFoundError(f"Missing matched clear image: {clear_image}")
        if clear_image.is_file() and base not in seen_bases:
            rows.append(
                {
                    **foggy_row,
                    "id": f"city/{base}/clear",
                    "image": str(clear_image),
                    "beta": 0.0,
                    "condition": "clear",
                }
            )
            seen_bases.add(base)


def prepare_cityscapes_manifest(
    root, output, with_detection_labels=False, splits=("train", "val")
):
    """Build foggy beta labels and optional, matched instance-box annotations."""
    root, output = Path(root).resolve(), Path(output)
    if output.exists():
        raise FileExistsError(f"{output} exists. Reuse it or choose a new output path.")
    image_root = find_foggy_cityscapes_image_root(root)
    rows = []
    for split in splits:
        rows.extend(
            create_cityscapes_split_rows(image_root, root, split, with_detection_labels)
        )
    if not rows:
        raise FileNotFoundError(
            f"No official Foggy Cityscapes images under {root}/leftImg8bit_foggy"
        )
    add_matched_clear_cityscapes_rows(rows, root, with_detection_labels)
    atomic_json(output, rows)
    return output

#### Tables, comparisons, and charts

In [ ]:
# fmt: off
#@title Supporting code: Tables, comparisons, and charts
# fmt: on
"""Saved-run analytics: tables, paired changes, per-class AP, density error and plots."""


BALANCED_METRICS = (
    "mAP50_95",
    "AP50",
    "AR100",
    "precision_at_025",
    "recall_at_025",
    "mean_ms",
    "p50_ms",
    "p95_ms",
    "serial_fps",
    "density_mean_ms",
)


def add_balanced_bin_rows(results):
    """Add equal-bin macro summaries without changing natural-distribution rows."""

    if "density_bin" not in results:

        def legacy_bin(value):
            if value in ("all", "unknown", "foggy", "balanced"):
                return value
            return classify_density_bin(float(value))

        results["density_bin"] = results.density.map(legacy_bin)
    if "distribution" not in results:
        results["distribution"] = "natural"
    candidates = results[
        ~results.density.isin(["all", "unknown", "balanced", "foggy"])
        & (results.density_bin != "unknown")
    ]
    summaries = []
    for keys, group in candidates.groupby(["model", "dataset", "arm", "seed"]):
        if group.density_bin.nunique() < 2:
            continue
        summary = group.iloc[0].to_dict()
        summary.update(
            density="balanced",
            classify_density_bin="balanced",
            distribution="balanced_bins",
            images=int(group.images.sum()),
            condition="mixed",
        )
        for column in BALANCED_METRICS:
            if column in group:
                summary[column] = float(
                    group.groupby("density_bin")[column].mean().mean()
                )
        summaries.append(summary)
    if summaries:
        results = pd.concat([results, pd.DataFrame(summaries)], ignore_index=True)
    return results


def collect_completed_run_outputs(output_directory):
    statuses, result_frames = [], []
    for run_directory in sorted(output_directory.iterdir()):
        if not run_directory.is_dir() or not (run_directory / STATUS_FILE).exists():
            continue
        status = json.loads((run_directory / STATUS_FILE).read_text())
        statuses.append(status)
        result_path = run_directory / RESULTS_FILE
        if status["status"] == "complete" and result_path.exists():
            result_frames.append(pd.read_csv(result_path, dtype={"density": str}))
    return statuses, result_frames


def save_training_status_reports(output_directory, statuses):
    pd.DataFrame(statuses).to_csv(output_directory / STATUS_TABLE_FILE, index=False)
    health = pd.DataFrame(statuses)
    health_columns = [
        name
        for name in (
            "model",
            "arm",
            "seed",
            "status",
            "planned_epochs",
            "epochs_trained",
            "best_epoch",
            "stop_reason",
        )
        if name in health
    ]
    health[health_columns].to_csv(output_directory / TRAINING_HEALTH_FILE, index=False)


def load_run_settings(output_directory):
    settings_path = output_directory / RUN_SETTINGS_FILE
    return (
        json.loads(settings_path.read_text(encoding="utf-8"))
        if settings_path.exists()
        else {}
    )


def add_recorded_run_metadata(results, settings):
    profile = settings.get(
        "profile", settings.get("config", {}).get("profile_name", PROFILE_FULL)
    )
    if isinstance(profile, dict):
        profile = profile.get("name", PROFILE_FULL)
    configuration = settings.get("config", {})
    if "profile" not in results:
        results["profile"] = profile
    if "evaluation_split" not in results:
        results["evaluation_split"] = configuration.get("evaluation_split", "test")
    if "planned_epochs" not in results:
        results["planned_epochs"] = configuration.get("epochs")
    if "epochs_trained" not in results:
        results["epochs_trained"] = np.nan
    if "stop_reason" not in results:
        results["stop_reason"] = "unknown"
    return results


def write_paired_comparison_tables(output_directory, results):
    keys = ["model", "dataset", "seed", "density"]
    metrics = ["mAP50_95", "mean_ms", "p95_ms"]
    controls = results[results.arm == ARM_RGB][keys + metrics]
    paired = results[results.arm != ARM_RGB].merge(
        controls,
        on=keys,
        how="inner",
        suffixes=("", "_control"),
        validate="many_to_one",
    )
    paired["delta_mAP"] = paired.mAP50_95 - paired.mAP50_95_control
    paired["delta_latency_ms"] = paired.mean_ms - paired.mean_ms_control
    paired.to_csv(output_directory / PAIRED_DELTA_FILE, index=False)
    export_hypothesis_tables(output_directory, paired)
    return paired


def write_seed_level_summary(output_directory, results):
    metric_names = [
        "mAP50_95",
        "AP50",
        "AR100",
        "precision_at_025",
        "recall_at_025",
        "mean_ms",
        "p95_ms",
        "serial_fps",
    ]
    summary = results.groupby(["dataset", "model", "arm", "density"])[metric_names].agg(
        ["mean", "std", "count"]
    )
    summary.to_csv(output_directory / SEED_SUMMARY_FILE)


def collect_density_checkpoint_errors(output_directory):
    rows = []
    for prediction_path in output_directory.glob("density-*/test-predictions.csv"):
        estimates = pd.read_csv(prediction_path)
        for dataset, dataset_estimates in estimates.groupby("dataset"):
            error = dataset_estimates.predicted_beta - dataset_estimates.target_beta
            rows.append(
                {
                    "experiment": prediction_path.parent.name,
                    "dataset": dataset,
                    "images": len(dataset_estimates),
                    "MAE_beta": float(error.abs().mean()),
                    "RMSE_beta": float(np.sqrt((error**2).mean())),
                }
            )
    return rows


def collect_detector_density_input_errors(output_directory, annotations):
    annotation_by_id = {row["id"]: row for row in annotations}
    rows = []
    for prediction_path in output_directory.glob(
        "*-predicted_density-*/*-predictions.jsonl"
    ):
        known_pairs = []
        with prediction_path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.endswith("\n"):
                    break
                prediction = json.loads(line)
                target_beta = annotation_by_id.get(prediction["id"], {}).get("beta")
                if target_beta is not None:
                    known_pairs.append((prediction["beta_input"], target_beta))
        if not known_pairs:
            continue
        errors = np.array([predicted - target for predicted, target in known_pairs])
        rows.append(
            {
                "experiment": prediction_path.parent.name,
                "dataset": prediction_path.stem.replace("-predictions", ""),
                "images": len(known_pairs),
                "MAE_beta": float(np.abs(errors).mean()),
                "RMSE_beta": float(np.sqrt((errors**2).mean())),
            }
        )
    return rows


def write_density_estimator_report(output_directory, annotations):
    rows = collect_density_checkpoint_errors(output_directory)
    rows.extend(collect_detector_density_input_errors(output_directory, annotations))
    pd.DataFrame(
        rows,
        columns=["experiment", "dataset", "images", "MAE_beta", "RMSE_beta"],
    ).to_csv(output_directory / DENSITY_REPORT_FILE, index=False)


def generate_study_analytics(output, rows=None):
    """Build saved tables and figures from completed comparison outputs."""
    output_directory = Path(output)
    if rows is None:
        rows = json.loads(
            (output_directory / INPUT_MANIFEST_FILE).read_text(encoding="utf-8")
        )
    statuses, result_frames = collect_completed_run_outputs(output_directory)
    save_training_status_reports(output_directory, statuses)
    if not result_frames:
        raise RuntimeError(
            f"No completed experiments under {output_directory}. Check status.csv; resume training first."
        )
    results = pd.concat(result_frames, ignore_index=True)
    results = add_recorded_run_metadata(results, load_run_settings(output_directory))
    results = add_balanced_bin_rows(results)
    results.to_csv(output_directory / RESULTS_FILE, index=False)
    paired = write_paired_comparison_tables(output_directory, results)
    write_seed_level_summary(output_directory, results)
    write_density_estimator_report(output_directory, rows)
    generate_comparison_figures(output_directory, results, paired)
    generate_hypothesis_figures(output_directory, results, paired, rows)
    return results


def comparison_plot_styles():
    import matplotlib

    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 10, "figure.dpi": 120})
    return {
        ARM_RGB: ("#33658A", "o"),
        ARM_DENSITY: ("#D17A22", "^"),
        ARM_ORACLE: ("#548235", "s"),
    }


def plot_overall_accuracy_and_latency(figures, dataset, overall, styles):
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    labels = []
    for position, ((model, arm), part) in enumerate(overall.groupby(["model", "arm"])):
        color, marker = styles[arm]
        labels.append(f"{model}\n{arm}")
        values = part.mAP50_95.to_numpy()
        axes[0].bar(position, np.nanmean(values), color=color, alpha=0.65)
        axes[0].scatter(
            np.full(len(values), position), values, color="black", marker=marker, s=20
        )
        axes[1].scatter(
            part.mean_ms,
            part.mAP50_95,
            color=color,
            marker=marker,
            label=f"{model}: {arm}",
        )
    axes[0].set_xticks(range(len(labels)), labels, rotation=25, ha="right")
    axes[0].set(
        ylabel="mAP@[.50:.95]",
        ylim=(0, 1),
        title="Bars: seed mean; points: individual seeds",
    )
    axes[1].set(
        xlabel="End-to-end mean latency (ms/image)",
        ylabel="mAP@[.50:.95]",
        ylim=(0, 1),
        title="Accuracy versus latency",
    )
    axes[1].legend(fontsize=7)
    profiles = ", ".join(sorted(str(value) for value in overall.profile.unique()))
    figure.suptitle(
        f"{dataset} | profile={profiles} | {overall.evaluation_split.iloc[0]} images"
    )
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-comparison.png", bbox_inches="tight")
    plt.close(figure)


def plot_balanced_density_accuracy(figures, dataset, balanced, styles):
    if balanced.empty:
        return
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(10, 5))
    labels = []
    for position, ((model, arm), part) in enumerate(balanced.groupby(["model", "arm"])):
        labels.append(f"{model}\n{arm}")
        color, marker = styles[arm]
        axis.bar(position, np.nanmean(part.mAP50_95), color=color, alpha=0.65)
        axis.scatter(
            np.full(len(part), position),
            part.mAP50_95,
            color="black",
            marker=marker,
            s=20,
        )
    axis.set_xticks(range(len(labels)), labels, rotation=25, ha="right")
    axis.set(
        ylabel="Equal-beta-bin macro mAP@[.50:.95]",
        ylim=(0, 1),
        title=f"{dataset}: balanced fog-density summary",
    )
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-balanced.png", bbox_inches="tight")
    plt.close(figure)


def plot_accuracy_by_fog_density(figures, dataset, frame, styles):
    numeric = frame[~frame.density.isin(["all", "unknown", "balanced", "foggy"])].copy()
    if numeric.empty:
        return
    import matplotlib.pyplot as plt

    numeric["beta"] = numeric.density.astype(float)
    figure, axis = plt.subplots(figsize=(9, 5))
    for (model, arm), part in numeric.groupby(["model", "arm"]):
        mean_accuracy = part.groupby("beta").mAP50_95.mean()
        axis.plot(
            mean_accuracy.index,
            mean_accuracy.values,
            marker=styles[arm][1],
            label=f"{model}: {arm}",
        )
    axis.set(
        xlabel="Known beta (m^-1)",
        ylabel="Mean mAP@[.50:.95]",
        ylim=(0, 1),
        title=f"{dataset}: accuracy by fog level",
    )
    axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-density.png")
    plt.close(figure)


def plot_per_class_accuracy(figures, dataset, overall):
    import matplotlib.pyplot as plt

    table = overall.groupby(["model", "arm"])[[f"AP_{name}" for name in CLASSES]].mean()
    figure, axis = plt.subplots(figsize=(10, max(3, 0.6 * len(table))))
    plotted = axis.imshow(
        table.to_numpy(), vmin=0, vmax=1, aspect="auto", cmap="viridis"
    )
    axis.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
    axis.set_yticks(
        range(len(table)), [f"{model}: {arm}" for model, arm in table.index]
    )
    axis.set_title(f"{dataset}: per-class AP (blank = no ground truth)")
    figure.colorbar(plotted, ax=axis, label="AP@[.50:.95]")
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-classes.png", bbox_inches="tight")
    plt.close(figure)


def plot_paired_accuracy_and_latency(figures, dataset, paired, styles):
    delta = paired[(paired.dataset == dataset) & (paired.density == "all")]
    if delta.empty:
        return
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels = []
    for position, ((model, arm), part) in enumerate(delta.groupby(["model", "arm"])):
        labels.append(f"{model}\n{arm}")
        for axis, column in zip(axes, ["delta_mAP", "delta_latency_ms"]):
            axis.scatter(
                np.full(len(part), position), part[column], marker=styles[arm][1]
            )
    for axis in axes:
        axis.axhline(0, color="black", linewidth=0.8)
        axis.set_xticks(range(len(labels)), labels, rotation=20, ha="right")
    axes[0].set_ylabel("Paired delta mAP (positive = better)")
    axes[1].set_ylabel("Paired delta latency ms (positive = slower)")
    figure.suptitle(f"{dataset}: matched seeds and test images; no significance claim")
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-deltas.png", bbox_inches="tight")
    plt.close(figure)


def plot_saved_training_loss(figures, history_file):
    import matplotlib.pyplot as plt

    history = pd.read_csv(history_file)
    if not {"train_loss", "val_loss"} <= set(history.columns):
        return
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.plot(history.epoch, history.train_loss, label="Train", marker="o")
    axis.plot(history.epoch, history.val_loss, label="Tuning", marker="^")
    axis.set(xlabel="Epoch", ylabel="Detector loss", title=history_file.parent.name)
    axis.legend()
    figure.tight_layout()
    figure.savefig(figures / f"{history_file.parent.name}-loss.png")
    plt.close(figure)


def find_training_history_files(output_directory):
    history_files = list(output_directory.glob("*/history.csv"))
    settings_path = output_directory / RUN_SETTINGS_FILE
    if settings_path.exists():
        source = (
            json.loads(settings_path.read_text())
            .get("config", {})
            .get("evaluation_source")
        )
        if source:
            history_files.extend(Path(source).glob("*/history.csv"))
    return history_files


def generate_comparison_figures(output, results, paired):
    """Create detector accuracy, latency, density, class, and loss charts."""
    figures = Path(output) / FIGURE_DIRECTORY
    figures.mkdir(exist_ok=True)
    styles = comparison_plot_styles()
    for dataset, frame in results.groupby("dataset"):
        overall = frame[frame.density == "all"]
        if overall.empty:
            continue
        plot_overall_accuracy_and_latency(figures, dataset, overall, styles)
        plot_balanced_density_accuracy(
            figures, dataset, frame[frame.density == "balanced"], styles
        )
        plot_accuracy_by_fog_density(figures, dataset, frame, styles)
        plot_per_class_accuracy(figures, dataset, overall)
        plot_paired_accuracy_and_latency(figures, dataset, paired, styles)
    for history_file in find_training_history_files(Path(output)):
        plot_saved_training_loss(figures, history_file)


def export_hypothesis_tables(output, paired):
    """Compare density versus control separately on clear and pooled fog images."""
    keys = ["dataset", "model", "arm", "seed"]
    clear = paired[paired.density == "0.0"][keys + ["delta_mAP", "images"]]
    fog = paired[paired.density == "foggy"][keys + ["delta_mAP", "images"]]
    effects = clear.merge(
        fog, on=keys, suffixes=("_clear", "_fog"), validate="one_to_one"
    )
    effects["fog_advantage"] = effects.delta_mAP_fog - effects.delta_mAP_clear
    effects["supports_direction_in_this_seed"] = (effects.delta_mAP_fog > 0) & (
        effects.delta_mAP_clear < 0
    )
    effects.to_csv(output / HYPOTHESIS_FILE, index=False)
    summaries = []
    for keys, group in effects.groupby(["dataset", "model", "arm"]):
        summary = dict(zip(["dataset", "model", "arm"], keys))
        summary["seeds"] = group.seed.nunique()
        summary["interpretation"] = "Exploratory; no seed uncertainty with one seed"
        for metric in ("delta_mAP_clear", "delta_mAP_fog", "fog_advantage"):
            values = group[metric].to_numpy()
            summary[metric + "_mean"] = float(values.mean())
            summary[metric + "_std"] = (
                float(values.std(ddof=1)) if len(values) > 1 else np.nan
            )
            lower = upper = np.nan
            if len(values) >= 3:
                samples = np.random.default_rng(42).choice(values, (5000, len(values)))
                lower, upper = np.percentile(samples.mean(axis=1), [2.5, 97.5])
                summary["interpretation"] = (
                    "95% bootstrap interval over training seeds; few seeds remain exploratory"
                )
            summary[metric + "_ci_low"] = lower
            summary[metric + "_ci_high"] = upper
        summaries.append(summary)
    pd.DataFrame(summaries).to_csv(output / HYPOTHESIS_SUMMARY_FILE, index=False)


def plot_clear_and_fog_model_comparison(figures, dataset, conditions):
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for axis, (density, title) in zip(axes, [("0.0", "Clear"), ("foggy", "Foggy")]):
        selected = conditions[conditions.density == density]
        table = selected.groupby(["model", "arm"]).mAP50_95.mean().unstack()
        arm_order = [arm for arm in (ARM_RGB, ARM_DENSITY, ARM_ORACLE) if arm in table]
        table[arm_order].rename(
            columns={
                ARM_RGB: "RGB control",
                ARM_DENSITY: "Predicted density",
                ARM_ORACLE: "Known density (oracle)",
            }
        ).plot.bar(ax=axis, color=["#33658A", "#D17A22", "#548235"][: len(arm_order)])
        axis.set(
            title=f"{title}: {selected.images.max()} images per model/arm",
            xlabel="Model",
            ylabel="Detection mAP (higher is better)",
            ylim=(0, 1),
        )
        axis.tick_params(axis="x", rotation=0)
        axis.legend(fontsize=7)
    figure.suptitle(f"{dataset}: same scenes and labels; bars average training seeds")
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-conditions.png", bbox_inches="tight")
    plt.close(figure)


def plot_density_arm_effects(figures, dataset, paired):
    delta = paired[(paired.dataset == dataset) & paired.density.isin(["0.0", "foggy"])]
    if delta.empty:
        return
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(12, 5))
    table = delta.groupby(["model", "arm", "density"]).delta_mAP.mean().unstack()
    table.rename(columns={"0.0": "Clear", "foggy": "Foggy"}).plot.bar(ax=axis)
    axis.set_xticklabels(
        [f"{model}\n{arm.replace('_', ' ')}" for model, arm in table.index], rotation=0
    )
    axis.axhline(0, color="black", linewidth=1)
    axis.set(
        ylabel="Density minus RGB control: delta mAP",
        xlabel="Model and density arm",
        title="Above zero helps; below zero hurts. One seed cannot establish significance.",
    )
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-effects.png", bbox_inches="tight")
    plt.close(figure)


def plot_condition_class_accuracy(figures, dataset, conditions):
    import matplotlib.pyplot as plt

    columns = [f"AP_{name}" for name in CLASSES]
    table = conditions.groupby(["density", "model", "arm"])[columns].mean()
    figure, axis = plt.subplots(figsize=(11, max(4, len(table) * 0.4)))
    heatmap = axis.imshow(table.to_numpy(), vmin=0, vmax=1, aspect="auto")
    axis.set_xticks(range(len(columns)), CLASSES, rotation=30)
    axis.set_yticks(range(len(table)), [" | ".join(key) for key in table.index])
    axis.set_title("Per-class AP by condition (blank = no labeled examples)")
    figure.colorbar(heatmap, ax=axis, label="AP: higher is better")
    figure.tight_layout()
    figure.savefig(figures / f"{dataset}-condition-classes.png", bbox_inches="tight")
    plt.close(figure)


def build_dataset_audit(rows):
    return pd.DataFrame(
        [
            {
                "dataset": row["dataset"],
                "split": row["split"],
                "bin": classify_density_bin(row["beta"]),
                "detection_labeled": row.get("detection_labeled", True),
                "pair_id": row.get("pair_id"),
                "objects": len(row["labels"]),
            }
            for row in rows
        ]
    )


def write_dataset_audit_tables(output, rows, figures):
    audit = build_dataset_audit(rows)
    if audit.empty:
        return
    audit.groupby(["dataset", "split", "bin", "detection_labeled"]).agg(
        images=("dataset", "size"),
        pairs=("pair_id", "nunique"),
        objects=("objects", "sum"),
    ).to_csv(output / DATASET_AUDIT_FILE)
    class_rows = [
        {
            "dataset": row["dataset"],
            "split": row["split"],
            "bin": classify_density_bin(row["beta"]),
            "class": CLASSES[label],
            "crowd": crowd,
        }
        for row in rows
        for label, crowd in zip(
            row["labels"], row.get("iscrowd", [0] * len(row["labels"]))
        )
    ]
    if class_rows:
        pd.DataFrame(class_rows).groupby(
            ["dataset", "split", "bin", "class", "crowd"]
        ).size().rename("objects").to_csv(output / CLASS_COVERAGE_FILE)
    plot_dataset_split_balance(figures, audit)


def plot_dataset_split_balance(figures, audit):
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(12, 5))
    audit.groupby(["dataset", "split", "bin"]).size().plot.bar(ax=axis)
    axis.set(
        ylabel="Images",
        xlabel="Dataset / split / fog level",
        title="Data balance: unknown fog condition is not clear",
    )
    figure.tight_layout()
    figure.savefig(figures / DATASET_AUDIT_FIGURE, bbox_inches="tight")
    plt.close(figure)


def generate_hypothesis_figures(output, results, paired, rows):
    """Draw the clear/fog hypothesis charts and data coverage audit."""
    output, figures = Path(output), Path(output) / FIGURE_DIRECTORY
    for dataset, frame in results.groupby("dataset"):
        conditions = frame[frame.density.isin(["0.0", "foggy"])]
        if conditions.empty:
            continue
        plot_clear_and_fog_model_comparison(figures, dataset, conditions)
        plot_density_arm_effects(figures, dataset, paired)
        plot_condition_class_accuracy(figures, dataset, conditions)
    write_dataset_audit_tables(output, rows, figures)
    generate_density_diagnostic_figures(output, rows)
    make_precision_recall_figures(output)


def collect_density_estimate_rows(output, annotations):
    annotation_by_id = {row["id"]: row for row in annotations}
    estimates = []
    for prediction_path in Path(output).glob(
        "*-predicted_density-*/*-predictions.jsonl"
    ):
        for line in prediction_path.read_text().splitlines(keepends=True):
            if not line.endswith("\n"):
                break
            prediction = json.loads(line)
            target_beta = annotation_by_id.get(prediction["id"], {}).get("beta")
            if target_beta is not None:
                estimates.append(
                    {
                        "experiment": prediction_path.parent.name,
                        "id": prediction["id"],
                        "target": target_beta,
                        "predicted": prediction["beta_input"],
                    }
                )
    return estimates


def plot_density_estimator_diagnostics(output, data):
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(data.target, data.predicted, s=8, alpha=0.3)
    axes[0].plot([0, BETA_SCALE], [0, BETA_SCALE], color="black")
    axes[0].set(
        xlabel="Known beta (m^-1)",
        ylabel="Predicted beta (m^-1)",
        title="Closer to diagonal = better density estimate",
    )
    data.boxplot(column="error", by="bin", ax=axes[1])
    axes[1].axhline(0, color="black")
    axes[0].set_xticks([0, 0.005, 0.01, BETA_SCALE])
    axes[1].set(title="Density error by fog level", ylabel="Predicted minus known beta")
    axes[2].hist(data.error, bins=25)
    axes[2].set(title="Density errors", xlabel="Beta error", ylabel="Predictions")
    axes[2].set_xticks([-0.01, -0.005, 0, 0.005, 0.01])
    figure.suptitle("Density quality is distinct from bounding-box detection quality")
    figure.tight_layout()
    figure.savefig(
        Path(output) / FIGURE_DIRECTORY / DENSITY_DIAGNOSTIC_FIGURE, bbox_inches="tight"
    )
    plt.close(figure)


def generate_density_diagnostic_figures(output, rows):
    """Save density-estimator errors separately from detector-quality metrics."""
    estimates = collect_density_estimate_rows(output, rows)
    if not estimates:
        return
    data = pd.DataFrame(estimates)
    data["error"] = data.predicted - data.target
    data["bin"] = data.target.map(classify_density_bin)
    data.to_csv(Path(output) / DENSITY_ESTIMATES_FILE, index=False)
    plot_density_estimator_diagnostics(output, data)


def make_precision_recall_figures(output):
    import matplotlib.pyplot as plt

    records = []
    for path in output.glob("*/*-precision-recall.json"):
        records.extend(json.loads(path.read_text()))
    if not records:
        return
    curves = pd.DataFrame(records)
    curves.to_json(output / "precision-recall.json", orient="records", indent=2)
    for dataset, frame in curves.groupby("dataset"):
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
        for ax, condition in zip(axes, ["clear", "foggy"]):
            for (model, arm), group in frame[frame.condition == condition].groupby(
                ["model", "arm"]
            ):
                precision = np.array(group.precision.tolist(), dtype=float)
                mean = np.nanmean(precision, axis=0)
                ax.plot(group.iloc[0].recall, mean, label=f"{model}: {arm}")
            ax.set(
                xlabel="Recall: fraction of labeled objects found",
                ylabel="Precision: fraction of reported objects correct",
                title=f"{condition.title()} images",
                xlim=(0, 1),
                ylim=(0, 1.02),
            )
            if ax.lines:
                ax.legend(fontsize=7)
        fig.suptitle(
            f"{dataset}: precision versus recall at IoU 0.50; class macro average"
        )
        fig.tight_layout()
        fig.savefig(
            output / FIGURE_DIRECTORY / f"{dataset}-precision-recall.png",
            bbox_inches="tight",
        )
        plt.close(fig)

#### Detector training and checkpoint recovery

#### Training settings and fog-density fitting

In [ ]:
# fmt: off
#@title Supporting code: Training settings and fog-density fitting (collapsed)
# fmt: on
def default_training_configuration(root):
    """Full-run settings for local command-line use, from the same notebook defaults."""
    root = Path(root).resolve()
    full = PROFILE_SETTINGS[PROFILE_FULL]
    return {
        **deepcopy(TRAINING_DEFAULTS),
        "output": str(root / "reports/fog/full-training-complete"),
        **{key: str(root / value) for key, value in MANIFEST_PATHS.items()},
        "checkpoints": {
            name: checkpoint if name == DETR_MODEL else str(root / checkpoint)
            for name, checkpoint in MODEL_SPECS.items()
        },
        "arms": list(DETECTION_ARMS),
        "seeds": list(FULL_RUN_SEEDS),
        "device": "cuda:0" if torch.cuda.is_available() else "cpu",
        "amp": True,
        "size": full["size"],
        "epochs": full["epochs"],
        "save_every": full["save_every"],
        "profile_name": PROFILE_FULL,
        "initialization": "existing_full_run",
        "evaluation_split": full["evaluation_split"],
        "early_stopping_patience": full["early_stopping_patience"],
    }


def summarize_cumulative_training_progress(progress, epochs, images_per_epoch):
    total_image_steps = epochs * images_per_epoch
    completed_image_steps = min(
        total_image_steps, progress.epoch * images_per_epoch + progress.step
    )
    return {
        "completed_epochs": progress.epoch,
        "planned_epochs": epochs,
        "saved_image_step_in_epoch": progress.step,
        "images_per_epoch": images_per_epoch,
        "completed_image_steps": completed_image_steps,
        "planned_image_steps": total_image_steps,
        "overall_training_percent": 100 * completed_image_steps / total_image_steps
        if total_image_steps
        else 0.0,
    }


def display_checkpoint_continuation(session, epochs, annotations, balanced=False):
    summary = summarize_cumulative_training_progress(
        session.progress, epochs, count_epoch_samples(annotations, balanced)
    )
    print(f"Checkpoint continuation: {session.checkpoint_directory}")
    print(
        f"Overall training: {summary['overall_training_percent']:.2f}% | "
        f"completed epochs {summary['completed_epochs']}/{epochs} | "
        f"saved image steps {summary['saved_image_step_in_epoch']}/{len(annotations)}"
    )
    if session.progress.history:
        print("Cumulative loss history, including previous sessions:")
        print(pd.DataFrame(session.progress.history).to_string(index=False))


def expected_profile_training_runs(configuration):
    expected = [
        (
            f"{DENSITY_RUN_PREFIX}-{SETTINGS['density_predictor']['seed']}",
            configuration["density_epochs"],
        )
    ]
    for seed in configuration["seeds"]:
        for model_name in configuration["checkpoints"]:
            for arm in configuration["arms"]:
                expected.append((f"{model_name}-{arm}-{seed}", configuration["epochs"]))
    return expected


def empty_training_progress_record(run_name, planned_epochs):
    return {
        "run": run_name,
        "completed_epochs": 0,
        "planned_epochs": planned_epochs,
        "saved_image_step_in_epoch": 0,
        "overall_training_percent": 0.0,
        "status": "not started",
    }


def read_saved_training_progress(checkpoint_path):
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    return DetectorTrainingProgress(
        **{
            key: saved[key]
            for key in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )


def describe_saved_run_progress(
    run_name, planned_epochs, output_directory, configuration
):
    record = empty_training_progress_record(run_name, planned_epochs)
    if run_name.startswith(f"{DENSITY_RUN_PREFIX}-"):
        run_directory = resolve_density_predictor_checkpoint(configuration).parent
    else:
        run_directory = output_directory / run_name
    checkpoint_path = run_directory / LAST_CHECKPOINT
    if checkpoint_path.exists():
        progress = read_saved_training_progress(checkpoint_path)
        manifest_path = output_directory / INPUT_MANIFEST_FILE
        if manifest_path.is_file():
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            training_rows = create_training_data_partitions(manifest, configuration)[
                "density_train"
                if run_name.startswith(f"{DENSITY_RUN_PREFIX}-")
                else "detector_train"
            ]
            balanced_sampling = (
                run_name.startswith(f"{DENSITY_RUN_PREFIX}-")
                and configuration.get("density_sampling", "natural") == "balanced_bins"
            )
            record.update(
                summarize_cumulative_training_progress(
                    progress,
                    planned_epochs,
                    count_epoch_samples(training_rows, balanced_sampling),
                )
            )
            record["status"] = (
                "training complete"
                if progress.epoch >= planned_epochs
                else "checkpoint saved"
            )
        else:
            record["status"] = "orphan checkpoint (manifest missing)"
    status_path = run_directory / STATUS_FILE
    if status_path.exists():
        record["status"] = json.loads(status_path.read_text())["status"]
    return record


def read_run_continuation_summary(configuration):
    """Show cumulative epoch and image-step progress for each saved training run."""
    output_directory = Path(configuration["output"])
    records = [
        describe_saved_run_progress(run_name, epochs, output_directory, configuration)
        for run_name, epochs in expected_profile_training_runs(configuration)
    ]
    return pd.DataFrame(records)


def fingerprint_data_inputs(rows):
    # Annotation/identity fingerprint plus image stat records; does not re-read every JPEG.
    value = hashlib.sha256()
    for row in sorted(rows, key=lambda r: r["id"]):
        stat = Path(row["image"]).stat()
        value.update(
            json.dumps([row, stat.st_size, stat.st_mtime_ns], sort_keys=True).encode()
        )
    return value.hexdigest()


def get_fog_model_training_data(rows):
    """Get the full beta-labeled train and validation rows for the fog model."""
    return {
        "density_train": [row for row in rows if row["split"] == "density_train"],
        "density_val": [
            row for row in rows if row["split"] == "val" and row["beta"] is not None
        ],
    }


def create_training_data_partitions(rows, config=None):
    """Use full beta data and profile-limit labeled detector training rows only."""
    density_splits = {
        "density_train": "density_train",
        "density_val": "val",
        "density_test": "test",
    }
    density_parts = {
        name: [row for row in rows if row["split"] == split and row["beta"] is not None]
        for name, split in density_splits.items()
    }
    profile = (config or {}).get("profile_scope")
    if profile and profile["name"] != PROFILE_FULL:
        rows = select_profile_training_rows(
            rows, profile["name"], profile["seed"], profile["training_image_cap"]
        )
        validation_cap = profile.get("validation_image_cap")
        if validation_cap:
            validation_rows = [
                row for row in rows if row.get("detector_split", row["split"]) == "val"
            ]
            rows = [
                row for row in rows if row.get("detector_split", row["split"]) != "val"
            ]
            rows += select_scene_safe_rows(
                validation_rows, validation_cap, profile["seed"], "val", "smoke:val"
            )
    parts = {
        name: [
            row
            for row in rows
            if row.get("detector_split", row["split"]) == name
            and row.get("detection_labeled", True)
            and (
                name not in ("detector_train", "val") or not any(row.get("iscrowd", []))
            )
        ]
        for name in ("detector_train", "val", "test")
    }
    parts.update(density_parts)
    evaluation_split = (
        "val" if config and config.get("evaluation_split") == "val" else "test"
    )
    parts["evaluation"] = parts[evaluation_split]
    parts["density_evaluation"] = density_parts[f"density_{evaluation_split}"]
    return parts


def validate_training_inputs(config, rows):
    """Check only the data needed for this detector comparison."""
    splits = create_training_data_partitions(rows, config)
    if config.get("evaluation_split", "test") not in VALID_EVALUATION_SPLITS:
        raise ValueError("Choose val or test as the evaluation split.")
    for name in ("detector_train", "val", "evaluation"):
        if not splits[name]:
            raise ValueError(f"No labeled {name} images are available.")
    if any(row["dataset"] == "nuscenes" for row in rows):
        path = Path(config["manifest"]).with_suffix(".coverage.json")
        coverage = json.loads(path.read_text()) if path.exists() else {}
        if coverage.get("complete") is not True and not config.get(
            "test_scope", {}
        ).get("enabled"):
            raise ValueError(f"nuScenes manifest is incomplete: {path}")
    if ARM_DENSITY in config["arms"]:
        checkpoint = config.get("density_checkpoint")
        if checkpoint and Path(checkpoint).is_file():
            saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
            fitted = {tuple(group) for group in saved["train_groups"]}
            held_out = {
                (row["dataset"], row["group"])
                for row in rows
                if row["split"] != "density_train"
            }
            if (
                saved["kind"] != "fog-density-v1"
                or saved["beta_scale"] != BETA_SCALE
                or not fitted
                or fitted & held_out
            ):
                raise ValueError(
                    "Fog-density checkpoint is incompatible with this study data."
                )
        elif (
            not splits["density_train"]
            or not splits["density_val"]
            or any(row["beta"] is None for row in splits["density_train"])
            or len({row["beta"] for row in splits["density_train"]}) < 2
        ):
            raise ValueError(
                "Fog-density training needs labeled fog images and held-out validation images."
            )
    if ARM_ORACLE in config["arms"] and any(
        row["beta"] is None
        for row in splits["detector_train"] + splits["val"] + splits["test"]
    ):
        raise ValueError(
            "Oracle comparison requires known fog density for every image."
        )


def save_torch(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


@dataclass
class DensityTrainingSession:
    model: object
    optimizer: object
    dataset_partitions: dict
    configuration: dict
    checkpoint_directory: Path
    seed: int
    progress: object


def restore_density_training_checkpoint(session, checkpoint_path):
    last_checkpoint = session.checkpoint_directory / LAST_CHECKPOINT
    checkpoint_path = last_checkpoint if last_checkpoint.is_file() else checkpoint_path
    if not checkpoint_path.is_file():
        return False
    saved = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.model.load_state_dict(saved["model"])
    if saved.get("optimizer"):
        session.optimizer.load_state_dict(saved["optimizer"])
    session.progress = DetectorTrainingProgress(
        epoch=saved.get("epoch", 0),
        step=saved.get("step", 0),
        best_loss=saved.get("best_loss", math.inf),
        running_sum=saved.get("running_sum", 0.0),
        running_count=saved.get("running_count", 0),
        history=saved.get("history", []),
    )
    display_checkpoint_continuation(
        session,
        session.configuration["density_epochs"],
        session.dataset_partitions["density_train"],
        balanced=session.configuration.get("density_sampling") == "balanced_bins",
    )
    return True


def save_density_training_checkpoint(session, improved=False):
    training_groups = sorted(
        {
            (annotation["dataset"], annotation["group"])
            for annotation in session.dataset_partitions["density_train"]
        }
    )
    payload = {
        "kind": "fog-density-v1",
        "beta_scale": BETA_SCALE,
        "model": session.model.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        "planned_epochs": session.configuration["density_epochs"],
        "density_sampling": session.configuration.get("density_sampling", "natural"),
        "seed": session.seed,
        **vars(session.progress),
        "train_groups": training_groups,
    }
    if improved:
        save_torch(session.checkpoint_directory / BEST_CHECKPOINT, payload)
    save_torch(session.checkpoint_directory / LAST_CHECKPOINT, payload)


def train_one_density_epoch(session):
    configuration = session.configuration
    progress = session.progress
    training_annotations = session.dataset_partitions["density_train"]
    image_order = select_epoch_sample_indices(
        training_annotations,
        session.seed,
        progress.epoch,
        balanced=configuration.get("density_sampling", "natural") == "balanced_bins",
    )
    session.model.train()
    description = f"Density seed={session.seed} epoch={progress.epoch + 1}"
    for image_step in tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    ):
        annotation = training_annotations[image_order[image_step]]
        image = load_image_tensor(
            annotation, configuration["size"], configuration["device"]
        )
        target_density = image.new_tensor([[annotation["beta"]]])
        session.optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(
            session.model(image) / BETA_SCALE, target_density / BETA_SCALE
        )
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite density loss")
        loss.backward()
        session.optimizer.step()
        progress.running_sum += float(loss.detach())
        progress.running_count += 1
        progress.step = image_step + 1
        if progress.step % configuration["save_every"] == 0:
            save_density_training_checkpoint(session)


def measure_density_validation_loss(session):
    session.model.eval()
    squared_errors = []
    with torch.inference_mode():
        for annotation in tqdm(
            session.dataset_partitions["density_val"],
            desc="Density tuning",
            mininterval=2,
        ):
            image = load_image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            predicted_density = session.model(image).item()
            squared_errors.append((predicted_density - annotation["beta"]) ** 2)
    return float(np.mean(squared_errors))


def record_completed_density_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_normalized_mse": progress.running_sum / progress.running_count,
            "val_beta_rmse": validation_loss**0.5,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_density_training_checkpoint(session, improved)
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / LOSS_HISTORY_FILE, index=False
    )


def should_stop_early(history, metric, patience):
    if not isinstance(patience, int) or patience <= 0 or len(history) <= patience:
        return False
    prior_best = min(row[metric] for row in history[:-patience])
    return all(row[metric] >= prior_best for row in history[-patience:])


def fit_fog_density_epochs(session, restored, progress_path):
    """Fit unfinished epochs and save an early-stop marker when needed."""
    configuration = session.configuration
    early_stopped = (
        progress_path.is_file()
        and json.loads(progress_path.read_text(encoding="utf-8")).get("reason")
        == "early_stopping"
    )
    if restored and (
        session.progress.epoch >= configuration["density_epochs"] or early_stopped
    ):
        print("Fog-density training is complete; using the saved best checkpoint.")
    elif restored:
        print("Continuing fog-density training from the saved epoch.")
    else:
        print("Starting fog-density training from default weights.")
    while (
        session.progress.epoch < configuration["density_epochs"] and not early_stopped
    ):
        train_one_density_epoch(session)
        validation_loss = measure_density_validation_loss(session)
        record_completed_density_epoch(session, validation_loss)
        if should_stop_early(
            session.progress.history,
            "val_beta_rmse",
            configuration.get("early_stopping_patience", 0),
        ):
            atomic_json(
                progress_path,
                {"epoch": session.progress.epoch, "reason": "early_stopping"},
            )
            early_stopped = True


def fit_fog_density_model(training_data, config, checkpoint_path, seed):
    """Restore saved fog-model training or initialize, fit, and freeze the model."""
    checkpoint_path = Path(checkpoint_path)
    directory = checkpoint_path.parent
    directory.mkdir(parents=True, exist_ok=True)
    set_reproducible_random_seed(seed)
    model = FogDensityRegressor().to(config["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    session = DensityTrainingSession(
        model,
        optimizer,
        training_data,
        config,
        directory,
        seed,
        DetectorTrainingProgress(),
    )
    restored = restore_density_training_checkpoint(session, checkpoint_path)
    fit_fog_density_epochs(session, restored, directory / PROGRESS_FILE)
    best_path = directory / BEST_CHECKPOINT
    last_path = directory / LAST_CHECKPOINT
    final_checkpoint = next(
        (path for path in (best_path, last_path, checkpoint_path) if path.is_file()),
        None,
    )
    if final_checkpoint is None:
        raise FileNotFoundError(
            f"Fog-density training produced no checkpoint in {directory}"
        )
    saved = torch.load(
        final_checkpoint, map_location=config["device"], weights_only=True
    )
    model.load_state_dict(saved["model"])
    return model.eval().requires_grad_(False)

#### Detector optimization and checkpoint recovery

In [ ]:
# fmt: off
#@title Supporting code: Detector optimization and checkpoint recovery (collapsed)
# fmt: on
@dataclass
class DetectorTrainingProgress:
    """Fields persisted in existing detector checkpoints; their names are stable."""

    epoch: int = 0
    step: int = 0
    best_loss: float = float("inf")
    running_sum: float = 0.0
    running_count: int = 0
    history: list = field(default_factory=list)


@dataclass
class DetectorTrainingSession:
    detector: object
    adapter: object
    density_estimator: object
    dataset_partitions: dict
    comparison_arm: str
    configuration: dict
    checkpoint_directory: Path
    seed: int
    optimizer: object
    gradient_scaler: object
    detector_parameters: list
    mixed_precision_enabled: bool
    progress: DetectorTrainingProgress = field(default_factory=DetectorTrainingProgress)


def create_detector_training_session(
    detector, adapter, density, parts, arm, config, directory, seed
):
    detector_parameters = [
        parameter
        for parameter in detector.model.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        [
            {
                "params": detector_parameters,
                "lr": config["learning_rates"][detector.name],
            },
            {"params": adapter.parameters(), "lr": config["adapter_lr"]},
        ],
        weight_decay=1e-4,
    )
    mixed_precision_enabled = config["amp"] and config["device"].startswith("cuda")
    gradient_scaler = torch.amp.GradScaler(
        "cuda", enabled=mixed_precision_enabled, init_scale=128.0
    )
    return DetectorTrainingSession(
        detector,
        adapter,
        density,
        parts,
        arm,
        config,
        directory,
        seed,
        optimizer,
        gradient_scaler,
        detector_parameters,
        mixed_precision_enabled,
    )


def restore_detector_training_checkpoint(session):
    checkpoint_path = session.checkpoint_directory / LAST_CHECKPOINT
    if not checkpoint_path.exists():
        return
    saved_checkpoint = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.optimizer.load_state_dict(saved_checkpoint["optimizer"])
    session.gradient_scaler.load_state_dict(saved_checkpoint["scaler"])
    session.progress = DetectorTrainingProgress(
        **{
            name: saved_checkpoint[name]
            for name in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )
    torch.set_rng_state(saved_checkpoint["rng"].cpu())
    if session.configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(
            [state.cpu() for state in saved_checkpoint["cuda_rng"]]
        )

    display_checkpoint_continuation(
        session,
        session.configuration["epochs"],
        session.dataset_partitions["detector_train"],
    )


def save_detector_training_checkpoint(session, improved=False, reason="periodic"):
    progress = session.progress
    payload = {
        "kind": "fog-detector-v1",
        "model": session.detector.model.state_dict(),
        "adapter": session.adapter.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        "scaler": session.gradient_scaler.state_dict(),
        **vars(progress),
        "rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all()
        if session.configuration["device"].startswith("cuda")
        else [],
        "checkpoint_reason": reason,
    }
    if improved:
        save_torch(session.checkpoint_directory / BEST_CHECKPOINT, payload)
    save_torch(session.checkpoint_directory / LAST_CHECKPOINT, payload)
    atomic_json(
        session.checkpoint_directory / PROGRESS_FILE,
        {
            "model": session.detector.name,
            "arm": session.comparison_arm,
            "seed": session.seed,
            "epoch": progress.epoch,
            "step": progress.step,
            "best_loss": float(progress.best_loss)
            if np.isfinite(progress.best_loss)
            else None,
            "reason": reason,
            **summarize_cumulative_training_progress(
                progress,
                session.configuration["epochs"],
                len(session.dataset_partitions["detector_train"]),
            ),
        },
    )


def calculate_finite_detector_loss(session, image, annotation, fog_density):
    with torch.autocast(
        device_type=image.device.type, enabled=session.mixed_precision_enabled
    ):
        loss = session.detector.training_loss(
            session.adapter(image, fog_density), annotation
        )
    if torch.isfinite(loss):
        return loss
    if session.mixed_precision_enabled:
        # Some examples overflow in float16; retry the same loss in float32.
        with torch.autocast(device_type=image.device.type, enabled=False):
            loss = session.detector.training_loss(
                session.adapter(image.float(), fog_density.float()), annotation
            )
        if torch.isfinite(loss):
            return loss
    raise ValueError(
        f"Nonfinite loss for {annotation['id']}; AMP fallback was also nonfinite"
    )


def apply_accumulated_detector_gradients(session):
    session.gradient_scaler.unscale_(session.optimizer)
    torch.nn.utils.clip_grad_norm_(
        session.detector_parameters + list(session.adapter.parameters()), 10
    )
    session.gradient_scaler.step(session.optimizer)
    session.gradient_scaler.update()
    session.optimizer.zero_grad(set_to_none=True)


def capture_detector_random_state(configuration):
    cpu_state = torch.get_rng_state()
    cuda_states = (
        torch.cuda.get_rng_state_all()
        if configuration["device"].startswith("cuda")
        else []
    )
    return cpu_state, cuda_states


def restore_detector_random_state(configuration, random_state):
    cpu_state, cuda_states = random_state
    torch.set_rng_state(cpu_state)
    if configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(cuda_states)


def accumulate_one_detector_image(session, annotation, accumulation_window_size):
    configuration = session.configuration
    image = load_image_tensor(
        annotation, configuration["size"], configuration["device"]
    )
    fog_density = get_density_feature(
        session.comparison_arm, image, annotation, session.density_estimator
    )
    loss = calculate_finite_detector_loss(session, image, annotation, fog_density)
    session.gradient_scaler.scale(loss / accumulation_window_size).backward()
    session.progress.running_sum += float(loss.detach())
    session.progress.running_count += 1


def checkpoint_completed_optimizer_update(session, next_image_step, image_count):
    apply_accumulated_detector_gradients(session)
    session.progress.step = next_image_step
    completed_window_count = next_image_step // session.configuration["accumulate"]
    checkpoint_due = completed_window_count % session.configuration["save_every"] == 0
    if checkpoint_due or next_image_step == image_count:
        save_detector_training_checkpoint(session)


def train_detector_microsteps(session, image_order):
    configuration = session.configuration
    progress = session.progress
    completed_loss_statistics = progress.running_sum, progress.running_count
    completed_random_state = capture_detector_random_state(configuration)
    description = f"{session.detector.name}/{session.comparison_arm} seed={session.seed} epoch={progress.epoch + 1}"
    image_steps = tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    )
    try:
        for image_step in image_steps:
            annotation = session.dataset_partitions["detector_train"][
                image_order[image_step]
            ]
            window_start = (image_step // configuration["accumulate"]) * configuration[
                "accumulate"
            ]
            window_size = min(
                configuration["accumulate"], len(image_order) - window_start
            )
            accumulate_one_detector_image(session, annotation, window_size)
            next_image_step = image_step + 1
            optimizer_boundary = next_image_step % configuration[
                "accumulate"
            ] == 0 or next_image_step == len(image_order)
            if optimizer_boundary:
                checkpoint_completed_optimizer_update(
                    session, next_image_step, len(image_order)
                )
                completed_loss_statistics = progress.running_sum, progress.running_count
                completed_random_state = capture_detector_random_state(configuration)
    except KeyboardInterrupt:
        # Replay the partial window with its original RNG state and loss statistics.
        session.optimizer.zero_grad(set_to_none=True)
        progress.running_sum, progress.running_count = completed_loss_statistics
        restore_detector_random_state(configuration, completed_random_state)
        save_detector_training_checkpoint(session, reason="interrupt")
        raise


def train_one_detector_epoch(session):
    session.detector.set_training_mode()
    session.adapter.train()
    training_image_count = len(session.dataset_partitions["detector_train"])
    image_order = random.Random(session.seed + session.progress.epoch).sample(
        range(training_image_count), training_image_count
    )
    session.optimizer.zero_grad(set_to_none=True)
    train_detector_microsteps(session, image_order)


def measure_detector_validation_loss(session):
    session.detector.model.eval()
    session.adapter.eval()
    validation_losses = []
    description = f"{session.detector.name}/{session.comparison_arm} tuning"
    with torch.no_grad():
        for annotation in tqdm(
            session.dataset_partitions["val"], desc=description, mininterval=2
        ):
            image = load_image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            fog_density = get_density_feature(
                session.comparison_arm, image, annotation, session.density_estimator
            )
            loss = calculate_finite_detector_loss(
                session, image, annotation, fog_density
            )
            validation_losses.append(float(loss))
    validation_loss = float(np.mean(validation_losses))
    if not np.isfinite(validation_loss):
        raise ValueError("Nonfinite validation loss")
    return validation_loss


def record_completed_detector_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_loss": progress.running_sum / progress.running_count,
            "val_loss": validation_loss,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_detector_training_checkpoint(session, improved, reason="epoch")
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / LOSS_HISTORY_FILE, index=False
    )


def load_best_detector_weights(session):
    saved_checkpoint = torch.load(
        session.checkpoint_directory / BEST_CHECKPOINT,
        map_location=session.configuration["device"],
        weights_only=True,
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.detector.model.eval()
    session.adapter.eval()


def fit_object_detector(
    detector, adapter, density, parts, arm, config, directory, seed
):
    """Resume, train, validate and checkpoint one detector comparison."""
    session = create_detector_training_session(
        detector, adapter, density, parts, arm, config, directory, seed
    )
    restore_detector_training_checkpoint(session)
    while session.progress.epoch < config["epochs"]:
        train_one_detector_epoch(session)
        validation_loss = measure_detector_validation_loss(session)
        record_completed_detector_epoch(session, validation_loss)
        if should_stop_early(
            session.progress.history,
            "val_loss",
            config.get("early_stopping_patience", 0),
        ):
            progress_path = directory / PROGRESS_FILE
            saved_progress = json.loads(progress_path.read_text(encoding="utf-8"))
            saved_progress["reason"] = "early_stopping"
            atomic_json(progress_path, saved_progress)
            break
    load_best_detector_weights(session)


def read_completed_prediction_records(predictions_path, ordered_annotations):
    completed_predictions = []
    if not predictions_path.exists():
        return completed_predictions
    with predictions_path.open("rb+") as stream_json_records:
        complete_record_end = 0
        while record_line := stream_json_records.readline():
            if not record_line.endswith(b"\n"):
                stream_json_records.truncate(complete_record_end)
                break
            completed_predictions.append(json.loads(record_line))
            complete_record_end = stream_json_records.tell()
    completed_ids = [prediction["id"] for prediction in completed_predictions]
    expected_ids = [
        annotation["id"]
        for annotation in ordered_annotations[: len(completed_predictions)]
    ]
    if completed_ids != expected_ids:
        raise ValueError(
            "Saved predictions do not match input order; use a new run directory"
        )
    return completed_predictions


def synchronize_inference_device(configuration):
    if configuration["device"].startswith("cuda"):
        torch.cuda.synchronize()


def predict_one_image_with_timing(
    detector, adapter, density_estimator, comparison_arm, annotation, configuration
):
    import time

    synchronize_inference_device(configuration)
    start_time = time.perf_counter()
    image = load_image_tensor(
        annotation, configuration["size"], configuration["device"]
    )
    synchronize_inference_device(configuration)
    before_density_time = time.perf_counter()
    fog_density = get_density_feature(
        comparison_arm, image, annotation, density_estimator
    )
    synchronize_inference_device(configuration)
    after_density_time = time.perf_counter()
    predictions = detector.predict_objects(adapter(image, fog_density), annotation)
    synchronize_inference_device(configuration)
    end_time = time.perf_counter()
    timing = {
        "total_ms": (end_time - start_time) * 1000,
        "density_ms": (after_density_time - before_density_time) * 1000,
        "detector_adapter_ms": (end_time - after_density_time) * 1000,
    }
    return predictions, float(fog_density.item()), timing


def evaluate_predictions_with_resume(
    detector, adapter, density, arm, rows, config, path
):
    """Keep valid prediction records, then evaluate only the remaining images."""
    completed_predictions = read_completed_prediction_records(path, rows)
    remaining_annotations = rows[len(completed_predictions) :]
    if remaining_annotations:
        for warmup_index in range(3):
            predict_one_image_with_timing(
                detector, adapter, density, arm, remaining_annotations[0], config
            )
    with path.open("a", encoding="utf-8") as stream_json_records:
        for annotation in tqdm(
            remaining_annotations, desc=f"Evaluate {detector.name}/{arm}", mininterval=2
        ):
            timings = []
            for repeat_index in range(config["repeats"]):
                predictions, fog_density, timing = predict_one_image_with_timing(
                    detector, adapter, density, arm, annotation, config
                )
                timings.append(timing)
            prediction_record = {
                "id": annotation["id"],
                "prediction": predictions,
                "beta_input": fog_density,
                "timings": timings,
            }
            stream_json_records.write(json.dumps(prediction_record) + "\n")
            stream_json_records.flush()
            completed_predictions.append(prediction_record)
    return completed_predictions


def export_trained_model(detector, adapter, directory, identity, config):
    """Write a compact inference artifact without optimizer, scaler or RNG state."""
    payload = {
        "kind": "fog-detector-export-v1",
        **identity,
        "checkpoint_source": config["checkpoints"][detector.name],
        "input_size": config["size"],
        "classes": list(CLASSES),
        "model": {
            key: value.detach().cpu()
            for key, value in detector.model.state_dict().items()
        },
        "adapter": {
            key: value.detach().cpu() for key, value in adapter.state_dict().items()
        },
    }
    save_torch(directory / MODEL_EXPORT_FILE, payload)


def export_saved_checkpoint(name, directory, identity, config):
    """Create the model-only artifact for a run completed by an older engine."""
    checkpoint = directory / BEST_CHECKPOINT
    if not checkpoint.exists() or (directory / MODEL_EXPORT_FILE).exists():
        return
    detector = ObjectDetector(
        name, config["checkpoints"][name], config["device"], trainable=False
    )
    adapter = FogAdapter().to(config["device"])
    saved = torch.load(checkpoint, map_location=config["device"], weights_only=True)
    detector.model.load_state_dict(saved["model"])
    adapter.load_state_dict(saved["adapter"])
    export_trained_model(detector, adapter, directory, identity, config)
    del detector, adapter

#### Evaluation and run identity

In [ ]:
# fmt: off
#@title Supporting code: Evaluation and run identity (collapsed)
# fmt: on
def read_detector_stop_metadata(directory, configuration):
    progress_path = directory / PROGRESS_FILE
    if not progress_path.exists():
        return {"epochs_trained": 0, "stop_reason": "unknown"}
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
    reason = progress.get("reason", "unknown")
    if reason == "epoch" and progress.get("epoch", 0) >= configuration["epochs"]:
        reason = "epoch_budget"
    return {"epochs_trained": progress.get("epoch", 0), "stop_reason": reason}


def build_run_signature(rows, configuration):
    selected_ids = sorted(row["id"] for row in rows)
    selection_hash = hashlib.sha256("\n".join(selected_ids).encode()).hexdigest()
    return {
        "engine": ENGINE_VERSION,
        "config": configuration,
        "inputs": fingerprint_data_inputs(rows),
        "profile": configuration.get("profile_name", PROFILE_FULL),
        "selection": {
            "schema": 1,
            "ids": selected_ids,
            "ids_sha256": selection_hash,
            "evaluation_split": configuration.get("evaluation_split", "test"),
        },
        "density_sha256": sha256_digest(configuration["density_checkpoint"])
        if configuration.get("density_checkpoint")
        else None,
        "source_revision": NOTEBOOK_REVISION,
        "versions": {
            package: importlib.metadata.version(package)
            for package in ("torch", "ultralytics", "transformers", "pycocotools")
        },
    }


def describe_run_identity_changes(previous, current):
    """Explain rejected continuation without dumping the selected image list."""
    changes = []
    for section in ("config", "versions"):
        before, after = previous.get(section, {}), current.get(section, {})
        for name in sorted(set(before) | set(after)):
            if before.get(name) != after.get(name):
                if isinstance(before.get(name), dict) and isinstance(
                    after.get(name), dict
                ):
                    fields = sorted(
                        key
                        for key in set(before[name]) | set(after[name])
                        if before[name].get(key) != after[name].get(key)
                    )
                    changes.append(
                        f"{section}.{name}: changed fields {', '.join(fields)}"
                    )
                else:
                    changes.append(
                        f"{section}.{name}: saved={str(before.get(name, 'not recorded'))[:120]}; requested={str(after.get(name, 'not recorded'))[:120]}"
                    )
    for name in ("engine", "density_sha256"):
        if previous.get(name) != current.get(name):
            changes.append(
                f"{name}: saved={previous.get(name)}; requested={current.get(name)}"
            )
    return "\n".join(changes[:15]) or "Checkpoint engine compatibility differs."


def validate_and_record_run_identity(rows, configuration, output_directory):
    """Protect saved checkpoints by matching settings and input data."""
    signature = build_run_signature(rows, configuration)
    settings_path = output_directory / RUN_SETTINGS_FILE
    has_checkpoints = any(output_directory.glob(f"*/{LAST_CHECKPOINT}"))
    if (
        has_checkpoints
        and not settings_path.exists()
        and configuration.get("profile_name") != PROFILE_FULL
    ):
        raise ValueError(
            "Existing checkpoints have no profile identity; choose a new output folder."
        )
    if settings_path.exists():
        previous = json.loads(settings_path.read_text())
        if previous != signature:
            immutable = ("config", "inputs", "density_sha256", "versions")
            compatible = all(
                previous.get(key) == signature.get(key) for key in immutable
            )
            previous_config = previous.get("config", {})
            full_run_can_resume = (
                configuration.get("profile_name") == PROFILE_FULL
                and configuration.get("initialization") == "existing_full_run"
                and all(
                    previous_config.get(key, FULL_RUN_DEFAULTS.get(key))
                    == configuration.get(key, FULL_RUN_DEFAULTS.get(key))
                    for key in FULL_RUN_IMMUTABLE_KEYS
                )
                and previous.get("inputs") == signature.get("inputs")
            )
            if full_run_can_resume:
                print(
                    "Continuing existing full-run checkpoints with recorded settings.",
                    flush=True,
                )
            elif compatible and previous.get("engine") in (
                ENGINE_VERSION - 1,
                ENGINE_VERSION,
            ):
                print(
                    "Migrating compatible checkpoint metadata to the current resumable engine.",
                    flush=True,
                )
            elif previous.get("inputs") != signature.get("inputs"):
                raise ValueError(
                    "The input manifest changed. Use a new output folder; the old checkpoint is preserved."
                )
            else:
                raise ValueError(
                    "Run configuration or package versions changed:\n"
                    + describe_run_identity_changes(previous, signature)
                    + "\nResume with saved settings and compatible packages, or use retrain=True. Existing checkpoints were preserved."
                )
    atomic_json(settings_path, signature)
    atomic_json(output_directory / INPUT_MANIFEST_FILE, rows)
    atomic_json(output_directory / INPUT_SELECTION_FILE, signature["selection"])


def evaluate_detector_comparison(
    detector,
    adapter,
    density_estimator,
    dataset_partitions,
    configuration,
    directory,
    identity,
    trainable_parameter_count,
):
    """Evaluate every held-out dataset and save its metric rows."""
    grouped = {}
    for row in dataset_partitions["evaluation"]:
        grouped.setdefault(row["dataset"], []).append(row)
    summaries = []
    for dataset in sorted(grouped):
        annotations = sorted(grouped[dataset], key=lambda row: row["id"])
        if configuration["device"].startswith("cuda"):
            torch.cuda.reset_peak_memory_stats()
        predictions = evaluate_predictions_with_resume(
            detector,
            adapter,
            density_estimator,
            identity["arm"],
            annotations,
            configuration,
            directory / f"{dataset}-predictions.jsonl",
        )
        summaries.extend(
            build_dataset_metric_rows(
                annotations,
                predictions,
                identity,
                directory,
                trainable_parameter_count,
                configuration["device"],
            )
        )
        save_dataset_precision_recall_curves(
            annotations, predictions, identity, directory
        )
    pd.DataFrame(summaries).to_csv(directory / RESULTS_FILE, index=False)


def build_dataset_metric_rows(
    annotations, predictions, identity, directory, trainable_parameter_count, device
):
    """Combine density-bin metrics with resource and checkpoint-size details."""
    if device.startswith("cuda"):
        peak_gpu_mb = torch.cuda.max_memory_allocated() / 2**20
    else:
        peak_gpu_mb = None
    checkpoint = Path(identity.get("source_checkpoint", directory / BEST_CHECKPOINT))
    checkpoint_mb = checkpoint.stat().st_size / 2**20
    return [
        {
            **identity,
            "dataset": annotations[0]["dataset"],
            **metrics,
            "peak_gpu_mb": peak_gpu_mb,
            "trainable_detector_parameters": trainable_parameter_count,
            "trained_checkpoint_mb": checkpoint_mb,
        }
        for metrics in summarize_detection_metrics_by_density(annotations, predictions)
    ]


def save_dataset_precision_recall_curves(annotations, predictions, identity, directory):
    """Save clear and foggy precision-recall curves for one dataset."""
    curves = []
    for condition in ("clear", "foggy"):
        indices = [
            index
            for index, row in enumerate(annotations)
            if row["beta"] is not None
            and (row["beta"] == 0 if condition == "clear" else row["beta"] > 0)
        ]
        if indices:
            curve = calculate_coco_detection_metrics(
                [annotations[index] for index in indices],
                [predictions[index] for index in indices],
                include_curve=True,
            )
            curves.append(
                {
                    **identity,
                    "dataset": annotations[0]["dataset"],
                    "condition": condition,
                    "precision": curve["precision_curve"],
                    "recall": curve["recall_curve"],
                }
            )
    atomic_json(
        directory / f"{annotations[0]['dataset']}-precision-recall.json", curves
    )


def validate_matched_evaluation(rows, source_rows, split, max_pairs=None):
    selected = [row for row in rows if row["split"] == split]
    if not selected:
        raise ValueError(f"No matched evaluation images in split {split}.")
    pairs = {}
    for row in selected:
        if not row.get("detection_labeled") or not row.get("annotation_source"):
            raise ValueError("Matched evaluation requires gtFine detection labels.")
        if not Path(row["annotation_source"]).is_file():
            raise FileNotFoundError(row["annotation_source"])
        if row.get("beta") is None or not row.get("pair_id"):
            raise ValueError("Matched evaluation requires known beta and pair_id.")
        pairs.setdefault(row["pair_id"], []).append(row)
    expected = set(DENSITY_BIN_UPPER_BOUNDS)
    for pair_id, variants in pairs.items():
        if (
            len(variants) != len(expected)
            or {row["beta"] for row in variants} != expected
        ):
            raise ValueError(f"Incomplete or duplicate clear/fog variants: {pair_id}")
        reference = variants[0]
        fields = ("boxes", "labels", "iscrowd", "width", "height", "group")
        if any(
            any(row[field] != reference[field] for field in fields) for row in variants
        ):
            raise ValueError(f"Mismatched ground truth: {pair_id}")
    fitted_splits = {"detector_train", "density_train"}
    if split == "test":
        fitted_splits.add("val")
    fitted_groups = {
        (row["dataset"], row["group"])
        for row in source_rows
        if row["split"] in fitted_splits
    }
    if any((row["dataset"], row["group"]) in fitted_groups for row in selected):
        raise ValueError("Evaluation scenes overlap checkpoint fitting/tuning scenes.")
    return sorted(
        limit_matched_evaluation_pairs(selected, pairs, max_pairs),
        key=lambda row: row["id"],
    )


def limit_matched_evaluation_pairs(selected, pairs, max_pairs):
    """Select a deterministic number of complete paired scenes when requested."""
    if max_pairs is None:
        return selected
    if not isinstance(max_pairs, int) or isinstance(max_pairs, bool) or max_pairs < 1:
        raise ValueError("max_pairs must be a positive integer or None.")
    chosen = set(
        sorted(pairs, key=lambda pair: hashlib.sha256(pair.encode()).hexdigest())[
            :max_pairs
        ]
    )
    return [row for row in selected if row["pair_id"] in chosen]


def run_checkpoint_evaluation(rows, configuration):
    """Evaluate saved best checkpoints while keeping the source run read-only."""
    source = Path(configuration["evaluation_source"]).resolve()
    output = Path(configuration["output"]).resolve()
    if source == output or source in output.parents or output in source.parents:
        raise ValueError("Evaluation output must be separate from the training run.")
    output.mkdir(parents=True, exist_ok=True)
    with evaluation_run_locks(source, output):
        selected, identity, checkpoints = prepare_checkpoint_evaluation(
            rows, configuration, source
        )
        validate_and_record_run_identity(selected, identity, output)
        for model, arm, seed, checkpoint in checkpoints:
            evaluate_saved_comparison(
                model, arm, seed, checkpoint, selected, configuration, output
            )
    return generate_study_analytics(output, selected), pd.read_csv(
        output / STATUS_TABLE_FILE
    )


@contextlib.contextmanager
def evaluation_run_locks(source, output):
    """Lock both folders so evaluation cannot race another run."""
    with contextlib.ExitStack() as stack:
        stack.enter_context(FileLock(str(source / RUN_LOCK_FILE), timeout=0))
        stack.enter_context(FileLock(str(output / RUN_LOCK_FILE), timeout=0))
        yield


def prepare_checkpoint_evaluation(rows, configuration, source):
    """Load source metadata and validate evaluation data and checkpoint selection."""
    source_settings = json.loads((source / RUN_SETTINGS_FILE).read_text())
    source_config = source_settings["config"]
    if configuration["size"] != source_config["size"]:
        raise ValueError("Use the source checkpoint's image size for evaluation.")
    source_rows = json.loads((source / INPUT_MANIFEST_FILE).read_text())
    selected = validate_matched_evaluation(
        rows,
        source_rows,
        configuration["evaluation_split"],
        configuration.get("evaluation_max_pairs"),
    )
    checkpoints, density_paths = select_checkpoint_evaluation_sources(
        source, source_config, configuration
    )
    identity = build_checkpoint_evaluation_identity(
        selected,
        configuration,
        source_config,
        checkpoints,
        density_paths,
    )
    return selected, identity, checkpoints


def select_checkpoint_evaluation_sources(source, source_config, configuration):
    """Find every requested trained detector and fog-density checkpoint."""
    checkpoints = []
    for seed in configuration["seeds"]:
        for model in configuration["checkpoints"]:
            for arm in configuration["arms"]:
                checkpoint = source / f"{model}-{arm}-{seed}" / BEST_CHECKPOINT
                if not checkpoint.is_file():
                    raise FileNotFoundError(f"Missing trained arm: {checkpoint}")
                if (
                    arm not in source_config["arms"]
                    or seed not in source_config["seeds"]
                ):
                    raise ValueError("Arm/seed was not trained in the source run.")
                checkpoints.append((model, arm, seed, checkpoint))
        if ARM_DENSITY in configuration["arms"]:
            path = Path(
                source_config.get("density_checkpoint")
                or source / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT
            )
            if not path.is_file():
                raise FileNotFoundError(path)
    density_paths = (
        [Path(source_config["density_checkpoint"])]
        if source_config.get("density_checkpoint")
        else [
            source / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT
            for seed in configuration["seeds"]
        ]
    )
    return checkpoints, density_paths


def build_checkpoint_evaluation_identity(
    rows, configuration, source_config, checkpoints, density_paths
):
    """Fingerprint checkpoints, fog predictor weights, and matched labels."""
    identity = dict(
        configuration,
        initialization="pretrained",
        source_training_config=source_config,
        checkpoint_sources={
            str(path): [path.stat().st_size, path.stat().st_mtime_ns]
            for _, _, _, path in checkpoints
        },
        evaluation_scope="matched_cityscapes_domain_shift",
    )
    identity["density_sources"] = {
        str(path): [path.stat().st_size, path.stat().st_mtime_ns]
        for path in density_paths
    }
    annotation_paths = {Path(row["annotation_source"]) for row in rows}
    identity["annotation_sources"] = {
        str(path): [path.stat().st_size, path.stat().st_mtime_ns]
        for path in annotation_paths
    }
    return identity


def evaluate_saved_comparison(model, arm, seed, checkpoint, rows, config, output):
    """Resume evaluation for one saved model, arm, and seed."""
    directory = output / f"{model}-{arm}-{seed}"
    directory.mkdir(exist_ok=True)
    status_path = directory / STATUS_FILE
    if is_saved_evaluation_complete(status_path):
        return
    identity = build_saved_evaluation_identity(model, arm, seed, checkpoint, config)
    try:
        atomic_json(status_path, {**identity, "status": "evaluating"})
        run_saved_detector_evaluation(
            model, arm, seed, checkpoint, rows, config, directory, identity
        )
        append_source_normal_use_metrics(checkpoint, directory)
        atomic_json(status_path, {**identity, "status": "complete"})
    except BaseException as error:
        record_saved_evaluation_error(status_path, identity, error)
        raise
    finally:
        release_evaluation_gpu_memory()


def is_saved_evaluation_complete(status_path):
    """Return whether this comparison already has a complete status record."""
    return (
        status_path.exists()
        and json.loads(status_path.read_text())["status"] == "complete"
    )


def build_saved_evaluation_identity(model, arm, seed, checkpoint, config):
    """Create the status metadata shared by each saved-checkpoint evaluation."""
    return {
        "model": model,
        "arm": arm,
        "seed": seed,
        "profile": config.get("profile_name", PROFILE_FULL),
        "evaluation_split": config["evaluation_split"],
        "source_checkpoint": str(checkpoint),
        "evaluation_scope": "matched_cityscapes_domain_shift",
    }


def run_saved_detector_evaluation(
    model, arm, seed, checkpoint, rows, config, directory, identity
):
    """Restore one detector and evaluate it on the selected matched images."""
    detector = adapter = density = None
    try:
        detector, adapter, density, saved, source_config = (
            load_saved_detector_for_evaluation(
                model,
                arm,
                seed,
                checkpoint,
                config,
            )
        )
        identity.update(
            planned_epochs=source_config["epochs"], best_epoch=saved.get("epoch")
        )
        identity.update(read_detector_stop_metadata(checkpoint.parent, source_config))
        parameters = sum(parameter.numel() for parameter in detector.model.parameters())
        evaluate_detector_comparison(
            detector,
            adapter,
            density,
            {"evaluation": rows},
            config,
            directory,
            identity,
            parameters,
        )
    finally:
        del detector, adapter, density


def load_saved_detector_for_evaluation(model, arm, seed, checkpoint, config):
    """Restore detector and adapter weights plus the optional fog predictor."""
    detector = ObjectDetector(
        model, config["checkpoints"][model], config["device"], trainable=True
    )
    adapter = FogAdapter().to(config["device"])
    saved = torch.load(checkpoint, map_location=config["device"], weights_only=True)
    source_config = json.loads(
        (checkpoint.parent.parent / RUN_SETTINGS_FILE).read_text()
    )["config"]
    detector.model.load_state_dict(saved["model"])
    adapter.load_state_dict(saved["adapter"])
    detector.model.eval()
    adapter.eval()
    density = None
    if arm == ARM_DENSITY:
        density_path = source_config.get("density_checkpoint") or (
            Path(config["evaluation_source"])
            / f"{DENSITY_RUN_PREFIX}-{seed}"
            / BEST_CHECKPOINT
        )
        density = load_fog_density_model(density_path, config["device"])
    return detector, adapter, density, saved, source_config


def record_saved_evaluation_error(status_path, identity, error):
    """Record a clear interrupted or failed status before preserving the error."""
    status = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
    atomic_json(
        status_path,
        {
            **identity,
            "status": status,
            "reason": f"{type(error).__name__}: {error}",
        },
    )


def release_evaluation_gpu_memory():
    """Release temporary model memory after each checkpoint evaluation."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def append_source_normal_use_metrics(checkpoint, directory):
    """Add the source run's fog-unknown nuScenes score beside fog results."""
    source_results = checkpoint.parent / RESULTS_FILE
    if not source_results.exists():
        return
    normal = pd.read_csv(source_results, dtype={"density": str})
    normal = normal[normal.dataset == "nuscenes"].copy()
    normal["evaluation_scope"] = "source_normal_use_fog_unknown"
    evaluation_results = pd.read_csv(directory / RESULTS_FILE, dtype={"density": str})
    pd.concat([evaluation_results, normal], ignore_index=True).to_csv(
        directory / RESULTS_FILE, index=False
    )

#### Detector comparison orchestration

In [ ]:
# fmt: off
#@title Supporting code: Detector comparison orchestration (collapsed)
# fmt: on
def run_detector_comparison(
    model_name,
    comparison_arm,
    seed,
    density_estimator,
    dataset_partitions,
    configuration,
    output_directory,
):
    directory = output_directory / f"{model_name}-{comparison_arm}-{seed}"
    directory.mkdir(exist_ok=True)
    identity = detector_comparison_identity(
        model_name, comparison_arm, seed, configuration
    )
    status_path = directory / STATUS_FILE
    if detector_comparison_is_complete(status_path):
        export_saved_checkpoint(model_name, directory, identity, configuration)
        return json.loads(status_path.read_text())
    detector, adapter = create_detector_comparison_models(
        model_name, seed, configuration
    )
    try:
        run_detector_training_and_evaluation(
            detector,
            adapter,
            density_estimator,
            dataset_partitions,
            configuration,
            directory,
            identity,
            status_path,
        )
    except BaseException as error:
        record_detector_comparison_error(status_path, identity, error)
        raise
    finally:
        release_detector_comparison_models(detector, adapter)
    return json.loads(status_path.read_text())


def detector_comparison_identity(model_name, comparison_arm, seed, configuration):
    return {
        "model": model_name,
        "arm": comparison_arm,
        "seed": seed,
        "profile": configuration.get("profile_name", PROFILE_FULL),
        "evaluation_split": configuration.get("evaluation_split", "test"),
        "planned_epochs": configuration["epochs"],
    }


def detector_comparison_is_complete(status_path):
    return (
        status_path.exists()
        and json.loads(status_path.read_text()).get("status") == "complete"
    )


def create_detector_comparison_models(model_name, seed, configuration):
    set_reproducible_random_seed(seed)
    detector = ObjectDetector(
        model_name,
        configuration["checkpoints"][model_name],
        configuration["device"],
        trainable=True,
    )
    set_reproducible_random_seed(seed)
    adapter = FogAdapter().to(configuration["device"])
    return detector, adapter


def run_detector_training_and_evaluation(
    detector,
    adapter,
    density_estimator,
    dataset_partitions,
    configuration,
    directory,
    identity,
    status_path,
):
    trainable_parameters = record_detector_training_started(
        status_path, identity, detector
    )
    fit_object_detector(
        detector,
        adapter,
        density_estimator,
        dataset_partitions,
        identity["arm"],
        configuration,
        directory,
        identity["seed"],
    )
    identity.update(read_detector_stop_metadata(directory, configuration))
    export_trained_model(detector, adapter, directory, identity, configuration)
    atomic_json(status_path, {**identity, "status": "evaluating"})
    evaluate_detector_comparison(
        detector,
        adapter,
        density_estimator,
        dataset_partitions,
        configuration,
        directory,
        identity,
        trainable_parameters,
    )
    atomic_json(status_path, {**identity, "status": "complete"})


def record_detector_training_started(status_path, identity, detector):
    trainable_parameters = sum(
        parameter.numel()
        for parameter in detector.model.parameters()
        if parameter.requires_grad
    )
    atomic_json(
        status_path,
        {
            **identity,
            "status": "training",
            "trainable_detector_parameters": trainable_parameters,
        },
    )
    return trainable_parameters


def record_detector_comparison_error(status_path, identity, error):
    status = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
    atomic_json(
        status_path,
        {**identity, "status": status, "reason": f"{type(error).__name__}: {error}"},
    )


def release_detector_comparison_models(detector, adapter):
    del detector, adapter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def prepare_detector_study(detector_rows, run_rows, config):
    """Validate both datasets, prepare detector splits, and initialize profile output."""
    if (
        config.get("overwrite_existing", False)
        and config.get("profile_name") != PROFILE_SETUP
    ):
        raise ValueError("overwrite_existing is only allowed for the setup profile.")
    validate_training_inputs(config, run_rows)
    partitions = create_training_data_partitions(detector_rows, config)
    output_directory = Path(config["output"])
    output_directory.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output_directory / RUN_LOCK_FILE), timeout=0):
        clear_setup_output_if_requested(config, output_directory, True)
        validate_and_record_run_identity(run_rows, config, output_directory)
    return partitions


def run_configured_detector_study(rows, config, fog_model, reset_setup=True):
    """Validate identity, resume detector comparisons, and produce reports."""
    output_directory = Path(config["output"])
    if (
        config.get("overwrite_existing", False)
        and config.get("profile_name") != PROFILE_SETUP
    ):
        raise ValueError("overwrite_existing is only allowed for the setup profile.")
    output_directory.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output_directory / RUN_LOCK_FILE), timeout=0):
        clear_setup_output_if_requested(config, output_directory, reset_setup)
        validate_and_record_run_identity(rows, config, output_directory)
        dataset_partitions = create_training_data_partitions(rows, config)
        statuses = run_comparison_matrix(
            dataset_partitions, config, output_directory, fog_model
        )
        return export_training_results(output_directory, rows, statuses)


def clear_setup_output_if_requested(config, output_directory, reset_setup):
    if not config.get("overwrite_existing", False) or not reset_setup:
        return
    for child in output_directory.iterdir():
        if child.name == RUN_LOCK_FILE:
            continue
        if not child.resolve().is_relative_to(output_directory.resolve()):
            raise ValueError("Setup cleanup target escapes its output folder.")
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()


def run_comparison_matrix(
    dataset_partitions,
    config,
    output_directory,
    fog_model,
    selected_model=None,
    selected_arm=None,
):
    statuses = []
    model_names = [selected_model] if selected_model else list(config["checkpoints"])
    arms = [selected_arm] if selected_arm else list(config["arms"])
    for seed_index, seed in enumerate(config["seeds"]):
        model_order = (
            model_names[seed_index % len(model_names) :]
            + model_names[: seed_index % len(model_names)]
        )
        for model_name in model_order:
            for arm in arms:
                statuses.append(
                    run_detector_comparison(
                        model_name,
                        arm,
                        seed,
                        fog_model,
                        dataset_partitions,
                        config,
                        output_directory,
                    )
                )
    return statuses


def export_training_results(output_directory, rows, statuses):

    return generate_study_analytics(output_directory, rows), pd.DataFrame(statuses)


def train_and_validate_detector(
    partitions, run_rows, config, fog_model, model_name, arm
):
    """Train or resume one model/arm and evaluate its configured held-out split."""
    output_directory = Path(config["output"])
    with FileLock(str(output_directory / RUN_LOCK_FILE), timeout=0):
        validate_and_record_run_identity(run_rows, config, output_directory)
        statuses = run_comparison_matrix(
            partitions,
            config,
            output_directory,
            fog_model,
            selected_model=model_name,
            selected_arm=arm,
        )
    return pd.DataFrame(statuses)

#### Notebook setup and high-level workflow steps

#### Profile and storage configuration

In [ ]:
# fmt: off
#@title Supporting code: Profile and storage configuration (collapsed)
# fmt: on
def relative_path(value):
    value = str(value)
    path = PurePosixPath(value)
    if (
        not value
        or path.is_absolute()
        or ".." in path.parts
        or "\\" in value
        or ":" in value
    ):
        raise ValueError(f"Expected a nonempty relative path: {value!r}")
    return Path(value)


def location_path(location):
    if location["kind"] == "drive":
        return (
            Path(SETTINGS["drive_mount"])
            / relative_path(SETTINGS["drive_subdir"])
            / relative_path(location["path"])
        )
    if location["kind"] == "path":
        path = Path(location["path"])
        if not path.is_absolute():
            raise ValueError("Filesystem locations must be absolute paths.")
        return path
    raise ValueError("Location kind must be 'drive' or 'path'.")


def single_name(value):
    path = relative_path(value)
    if len(path.parts) != 1 or value in (".", ".."):
        raise ValueError(f"Expected a single filename or directory name: {value!r}")
    return value


def validate_study_settings():
    """Check the notebook settings once before using paths or loading data."""
    global ROOT
    profile = SETTINGS["profile"]
    name = profile["name"]
    if name not in PROFILE_NAMES:
        raise ValueError(f"profile.name must be one of {PROFILE_NAMES}")
    expected = "existing_full_run" if name == PROFILE_FULL else "pretrained"
    if profile["initialization"] != expected:
        raise ValueError(f"The {name} profile must use {expected} initialization.")
    if not isinstance(SETTINGS.get("retrain", False), bool):
        raise ValueError("retrain must be True or False.")
    predictor = SETTINGS["density_predictor"]
    single_name(predictor["source_run"])
    if predictor["seed"] < 0:
        raise ValueError("density_predictor.seed must be nonnegative.")
    for key in ("run_name", "export_name", "packages_file"):
        single_name(SETTINGS["output"][key])

    data = SETTINGS["data"]
    if data["mode"] not in {"zip", "directory"}:
        raise ValueError("Data mode must be zip or directory.")
    if data["minimum_free_gib"] < 0:
        raise ValueError("minimum_free_gib cannot be negative.")
    single_name(data["archive"])
    if (
        not Path(SETTINGS["runtime_root"]).is_absolute()
        or not Path(SETTINGS["drive_mount"]).is_absolute()
    ):
        raise ValueError("runtime_root and drive_mount must be absolute.")
    if data["low_disk"] and data["location"]["kind"] != "drive":
        raise ValueError("Low-disk mode requires data in Drive with no URL.")
    ROOT = Path(SETTINGS["runtime_root"])


def connect_and_resolve_study_storage():
    global DATA_ROOT, RESOLVED_PATHS, TRAINING_CONFIGURATION
    data_settings = SETTINGS["data"]
    # Resolve paths; authentication already ran before setup.
    data_location = location_path(data_settings["location"])
    output_location = location_path(SETTINGS["output"]["location"])
    if any(
        loc["kind"] == "drive"
        for loc in (data_settings["location"], SETTINGS["output"]["location"])
    ):
        drive_root = Path(SETTINGS["drive_mount"]) / SETTINGS["drive_subdir"]
        if not drive_root.is_dir():
            raise RuntimeError(
                "Google Drive is disconnected. Rerun the connection cell near the top."
            )
    ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    DATA_ROOT = data_location if data_settings["mode"] == "directory" else ROOT
    RESOLVED_PATHS = {
        "data_location": data_location,
        "data_root": DATA_ROOT,
        "archive": data_location / data_settings["archive"],
        "output_root": output_location,
        "run": output_location / SETTINGS["output"]["run_name"],
        "export": output_location / SETTINGS["output"]["export_name"],
        "marker": ROOT / relative_path(data_settings["extraction_marker"]),
        "packages": ROOT / SETTINGS["output"]["packages_file"],
        "city_images": DATA_ROOT / relative_path(data_settings["city_images"]),
        "nuscenes": DATA_ROOT / relative_path(data_settings["nuscenes"]),
    }
    RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
    TRAINING_CONFIGURATION = dict(SETTINGS["training"])


def apply_subset_execution_settings():
    """Apply smoke-test or selected-profile limits to the shared run config."""
    if SETTINGS.get("test", {}).get("enabled"):
        apply_smoke_test_profile()
    profile = SETTINGS["profile"]
    if profile["name"] == PROFILE_FULL:
        apply_full_profile_settings(profile)
    else:
        apply_partial_profile_settings(profile)


def apply_smoke_test_profile():
    """Build a tiny profile for bounded end-to-end notebook verification."""
    test = SETTINGS["test"]
    smoke = dict(PROFILE_SETTINGS[PROFILE_SETUP])
    smoke.update(
        name=PROFILE_SETUP,
        run_suffix=f"-{PROFILE_SETTINGS[PROFILE_SETUP]['run_name']}",
        initialization="pretrained",
        models=test.get("models", smoke["models"]),
        seed=test.get("seed", smoke["seed"]),
        epochs=test.get("epochs", smoke["epochs"]),
        size=test.get("image_size", smoke["size"]),
    )
    images = test.get("images_per_stratum", 4)
    smoke["training_image_cap"] = max(6, images * 6)
    smoke["validation_image_cap"] = max(3, images * 3)
    SETTINGS["profile"] = smoke


def apply_partial_profile_settings(profile):
    """Apply partial-run limits and direct artifacts to their profile output."""
    single_name(profile["run_suffix"])
    output = RESOLVED_PATHS["output_root"]
    RESOLVED_PATHS["run"] = output / SETTINGS["output"]["run_name"]
    RESOLVED_PATHS["export"] = output / SETTINGS["output"]["export_name"]
    RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
    TRAINING_CONFIGURATION.update(
        epochs=profile["epochs"],
        density_epochs=profile.get(
            "density_epochs", TRAINING_CONFIGURATION["density_epochs"]
        ),
        size=profile["size"],
        save_every=profile["save_every"],
        seeds=[profile["seed"]],
        early_stopping_patience=profile["early_stopping_patience"],
        profile_name=profile["name"],
        initialization="pretrained",
        evaluation_split=profile["evaluation_split"],
        density_sampling=profile.get("density_sampling", "natural"),
        overwrite_existing=profile.get("overwrite_existing", False),
    )
    TRAINING_CONFIGURATION["profile_scope"] = dict(profile)
    TRAINING_CONFIGURATION["test_scope"] = {
        "enabled": True,
        "profile": profile["name"],
        "scientific_results": profile["name"] != PROFILE_SETUP,
    }


def apply_full_profile_settings(profile):
    """Preserve the configured full-run checkpoint continuation settings."""
    TRAINING_CONFIGURATION.update(
        profile_name=PROFILE_FULL,
        initialization="existing_full_run",
        evaluation_split="test",
        early_stopping_patience=0,
        density_sampling=profile.get("density_sampling", "natural"),
    )


def resolve_manifest_and_model_paths():
    data_settings = SETTINGS["data"]
    selected_models = SETTINGS["profile"]["models"]
    TRAINING_CONFIGURATION.update(
        {
            key: str(DATA_ROOT / relative_path(value))
            for key, value in data_settings["manifests"].items()
        }
    )
    TRAINING_CONFIGURATION["output"] = str(RESOLVED_PATHS["run"])
    TRAINING_CONFIGURATION["checkpoints"] = {
        name: value if name == DETR_MODEL else str(ROOT / Path(value))
        for name, value in SETTINGS["models"].items()
        if name in selected_models
    }
    if TRAINING_CONFIGURATION["density_checkpoint"]:
        TRAINING_CONFIGURATION["density_checkpoint"] = str(
            ROOT / Path(TRAINING_CONFIGURATION["density_checkpoint"])
        )


def resolve_study_configuration(settings):
    """Validate settings, connect storage and resolve one experiment configuration."""
    global SETTINGS
    SETTINGS = settings
    validate_study_settings()
    connect_and_resolve_study_storage()
    saved_configuration = restore_existing_profile_settings()
    apply_training_profile(saved_configuration)
    resolve_default_export_path()
    allocate_fresh_retrain_output()
    configure_density_predictor_checkpoint()
    return TRAINING_CONFIGURATION, RESOLVED_PATHS


def apply_training_profile(saved_configuration):
    """Resolve new settings unless this output folder owns a saved experiment."""
    apply_subset_execution_settings()
    resolve_manifest_and_model_paths()
    if saved_configuration is None:
        return
    TRAINING_CONFIGURATION.clear()
    TRAINING_CONFIGURATION.update(saved_configuration)


def resolve_default_export_path():
    """Keep a default report folder beside the selected run folder."""
    expected = SETTINGS["profile"]["run_name"] + "-results"
    if SETTINGS["output"]["export_name"] != expected:
        return
    run = RESOLVED_PATHS["run"]
    RESOLVED_PATHS["export"] = run.with_name(run.name + "-results")


def allocate_fresh_retrain_output():
    """Give retraining its own folder so prior checkpoints remain recoverable."""
    if (
        not SETTINGS.get("retrain", False)
        or SETTINGS["profile"]["name"] == PROFILE_SETUP
    ):
        return
    run = RESOLVED_PATHS["run"]
    output = run.with_name(run.name + "-retrain-" + uuid4().hex[:12])
    output.mkdir(parents=True, exist_ok=False)
    TRAINING_CONFIGURATION.update(
        output=str(output), initialization="pretrained", density_checkpoint=None
    )
    RESOLVED_PATHS["run"] = output
    RESOLVED_PATHS["export"] = output.with_name(output.name + "-results")
    print("Fresh training requested. Previous run preserved. New output:", output)


def configure_density_predictor_checkpoint():
    """Load a completed estimator for new detector profiles; preserve full resumes."""
    options = SETTINGS["density_predictor"]
    configured = str(options.get("checkpoint", "")).strip()
    if configured:
        checkpoint = Path(configured)
        if not checkpoint.is_absolute():
            checkpoint = ROOT / checkpoint
    elif (
        SETTINGS["profile"]["name"] == PROFILE_FULL
        and (RESOLVED_PATHS["run"] / RUN_SETTINGS_FILE).is_file()
    ):
        return
    else:
        checkpoint = (
            RESOLVED_PATHS["output_root"]
            / options["source_run"]
            / f"{DENSITY_RUN_PREFIX}-{options['seed']}"
            / BEST_CHECKPOINT
        )
    TRAINING_CONFIGURATION["density_checkpoint"] = str(checkpoint.resolve())

#### Data preparation and readiness checks

In [ ]:
# fmt: off
#@title Supporting code: Data loading and preparation (collapsed)
# fmt: on
def restore_existing_profile_settings():
    """An existing output folder defines the experiment; new defaults do not."""
    path = RESOLVED_PATHS["run"] / RUN_SETTINGS_FILE
    if (
        SETTINGS.get("retrain", False)
        or SETTINGS.get("test", {}).get("enabled", False)
        or SETTINGS["profile"]["name"] == PROFILE_SETUP
        or not path.exists()
    ):
        return None
    recorded = json.loads(path.read_text(encoding="utf-8"))["config"]
    if recorded.get("profile_name", PROFILE_FULL) != SETTINGS["profile"]["name"]:
        raise ValueError("Saved output belongs to another profile.")
    SETTINGS["profile"].update(recorded.get("profile_scope", {}))
    SETTINGS["profile"]["density_sampling"] = recorded.get(
        "density_sampling", "natural"
    )
    SETTINGS["profile"]["overwrite_existing"] = False
    recorded["output"] = str(RESOLVED_PATHS["run"])
    if SETTINGS["profile"]["name"] == PROFILE_FULL:
        recorded.setdefault("profile_name", PROFILE_FULL)
        recorded.setdefault("initialization", "existing_full_run")
    print("Using this run's recorded training settings (including density sampling).")
    return recorded


def should_reuse_completed_training(configuration):
    """Check saved experiment statuses without opening images or model weights."""
    if SETTINGS["profile"]["name"] == PROFILE_SETUP or SETTINGS.get("retrain", False):
        return False
    output = Path(configuration["output"])
    if not (output / RUN_SETTINGS_FILE).is_file():
        return False
    directories = [
        output / f"{model}-{arm}-{seed}"
        for model in configuration["checkpoints"]
        for arm in configuration["arms"]
        for seed in configuration["seeds"]
    ]
    return bool(directories) and all(
        (directory / RESULTS_FILE).is_file()
        and (directory / STATUS_FILE).is_file()
        and json.loads((directory / STATUS_FILE).read_text()).get("status")
        == "complete"
        for directory in directories
    )


def check_free_space(required=0):
    free = shutil.disk_usage(ROOT).free
    reserve = int(SETTINGS["data"]["minimum_free_gib"] * 2**30)
    if required + reserve > free:
        raise OSError(
            f"Need {(required + reserve) / 2**30:.2f} GiB free; available {free / 2**30:.2f} GiB."
        )


def prepare_fog_data():
    if SETTINGS["data"]["mode"] == "zip":
        dataset_archive = find_or_download_dataset_archive()
        validate_and_extract_full_dataset_archive(dataset_archive)
    verify_prepared_manifest_files()


def find_or_download_dataset_archive():
    bundle = RESOLVED_PATHS["archive"]
    if not bundle.is_file():
        import gdown

        bundle.parent.mkdir(parents=True, exist_ok=True)
        partial = bundle.with_suffix(bundle.suffix + ".part")
        gdown.download(SETTINGS["data"]["url"], str(partial), quiet=False, fuzzy=True)
        partial.replace(bundle)
    return bundle


def validate_and_extract_full_dataset_archive(bundle):
    data_settings = SETTINGS["data"]
    info = bundle.stat()
    identity = {
        "path": str(bundle.resolve()),
        "size": info.st_size,
        "modified_ns": info.st_mtime_ns,
        "manifests": data_settings["manifests"],
    }
    marker = RESOLVED_PATHS["marker"]
    try:
        previous = json.loads(marker.read_text()) if marker.exists() else None
    except (ValueError, OSError):
        previous = None
    if previous != identity:
        with zipfile.ZipFile(bundle) as archive:
            members = archive.infolist()
            required_manifests = set(data_settings["manifests"].values())
            if not required_manifests.issubset(archive.namelist()):
                raise ValueError(
                    f"Archive must contain prepared manifests: {sorted(required_manifests)}"
                )
            for archive_member_name in members:
                destination = (ROOT / archive_member_name.filename).resolve()
                if (
                    not destination.is_relative_to(ROOT.resolve())
                    or not archive_member_name.filename.startswith(
                        tuple(data_settings["allowed_archive_roots"])
                    )
                    or stat.S_ISLNK(archive_member_name.external_attr >> 16)
                ):
                    raise ValueError(
                        f"Unsafe or unexpected archive member: {archive_member_name.filename}"
                    )
            check_free_space(
                sum(archive_member_name.file_size for archive_member_name in members)
            )
            archive.extractall(ROOT)
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps(identity), encoding="utf-8")


def verify_prepared_manifest_files():
    for relative_manifest in SETTINGS["data"]["manifests"].values():
        path = RESOLVED_PATHS["data_root"] / relative_path(relative_manifest)
        if not path.is_file():
            raise FileNotFoundError(f"Missing prepared manifest: {path}")
    check_free_space()
    print("Prepared data:", RESOLVED_PATHS["data_root"])


def display_dataset_split_summary(annotations, configuration):
    if not annotations:
        return
    frame = pd.DataFrame(annotations)
    display(
        frame.groupby(["dataset", "split"]).agg(
            images=("id", "size"),
            scenes=("group", "nunique"),
            known_beta=("beta", "count"),
        )
    )
    coverage_path = Path(configuration["manifest"]).with_suffix(".coverage.json")
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(
            pd.DataFrame(
                [
                    {
                        "expected": coverage["expected_images"],
                        "available": coverage["available_images"],
                        "missing": len(coverage["missing_images"]),
                        "complete": coverage["complete"],
                    }
                ]
            )
        )


def load_study_manifest(path, normalize_timestamps=True):
    """Load one dataset manifest and stabilize image times used in run fingerprints."""
    rows = load_manifest(path)
    if normalize_timestamps:
        for row in rows:
            os.utime(
                row["image"],
                ns=(STUDY_IMAGE_TIMESTAMP_NS, STUDY_IMAGE_TIMESTAMP_NS),
            )
    return rows

#### Study workflow and result display

In [ ]:
# fmt: off
#@title Supporting code: Study workflow and result display (collapsed)
# fmt: on


def load_labeled_clear_fog_data():
    """Select the configured held-out split from the shared labeled manifest."""
    prepare_cityscapes_data()
    rows = load_labeled_cityscapes_data()
    split = SETTINGS["evaluation"]["split"]
    return [row for row in rows if row["split"] == split]


def run_saved_fog_comparison(rows):
    """Evaluate saved detectors on the supplied matched clear/fog images."""
    options = SETTINGS["evaluation"]
    source = Path(options["source_run"] or TRAINING_CONFIGURATION["output"])
    source_config = json.loads((source / RUN_SETTINGS_FILE).read_text())["config"]
    output = RESOLVED_PATHS["output_root"] / options["output_name"]
    if source == output or source in output.parents or output in source.parents:
        raise ValueError("Evaluation output must be separate from the training run.")
    evaluation = dict(source_config)
    evaluation.update(
        output=str(output),
        evaluation_source=str(source),
        evaluation_split=options["split"],
        evaluation_max_pairs=options["max_pairs"],
        device=TRAINING_CONFIGURATION["device"],
        amp=TRAINING_CONFIGURATION["amp"],
    )
    checkpoint = resolve_density_predictor_checkpoint(evaluation)
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Train the selected profile first: {checkpoint}")
    load_fog_density_model(checkpoint, evaluation["device"])
    return run_checkpoint_evaluation(rows, evaluation)


def run_optional_synthetic_benchmark(enabled):
    """Measure a synthetic batch-64 ceiling without changing study results."""
    import time

    import torch
    import torchvision
    from torch.utils.data import DataLoader, TensorDataset

    can_run = enabled and torch.cuda.is_available()
    can_run = can_run and SETTINGS["profile"]["name"] == PROFILE_FULL
    if not can_run:
        print("Synthetic batch-64 benchmark skipped for this profile or CUDA state.")
        return
    print(f"Benchmarking on: {torch.cuda.get_device_name(0)}")
    image_count, batch_size = BENCHMARK_IMAGES, BENCHMARK_BATCH_SIZE
    images = torch.randn(image_count, 3, 640, 640, device="cuda", dtype=torch.float16)
    labels = torch.randn(image_count, 3, 640, 640, device="cuda", dtype=torch.float16)
    loader = DataLoader(TensorDataset(images, labels), batch_size=batch_size)
    model = torchvision.models.resnet50(weights=None).to("cuda").half()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(5):
        model(images[:batch_size])
    torch.cuda.synchronize()
    started = time.perf_counter()
    for batch_images, _ in loader:
        optimizer.zero_grad()
        model(batch_images).sum().backward()
        optimizer.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    print(f"Processed {image_count} images in {elapsed:.2f} seconds.")
    print(f"Synthetic benchmark throughput: {image_count / elapsed:.2f} images/second.")


def display_detection_accuracy_table(results):
    if results.empty:
        print("No completed runs yet.")
        return
    columns = [
        "dataset",
        "model",
        "arm",
        "seed",
        "density",
        "images",
        "mAP50_95",
        "AP50",
        "precision_at_025",
        "recall_at_025",
        "AR100",
    ]
    print(
        "Detection quality: higher is better. Clear = beta 0; foggy = pooled positive beta."
    )
    display(
        results.loc[
            results.density.isin(["all", "balanced", "0.0", "foggy"]), columns
        ].round(4)
    )


def display_inference_latency_table(results):
    print("Inference cost: lower milliseconds is better; higher FPS is better.")
    display(
        results.loc[
            results.density == "all",
            [
                "dataset",
                "model",
                "arm",
                "seed",
                "mean_ms",
                "p50_ms",
                "p95_ms",
                "serial_fps",
                "density_mean_ms",
            ],
        ].round(2)
    )


def display_fog_hypothesis_tables(output):
    paired = pd.read_csv(Path(output) / PAIRED_DELTA_FILE, dtype={"density": str})
    display(
        paired.loc[
            paired.density.isin(["all", "balanced"]),
            ["model", "dataset", "seed", "arm", "delta_mAP", "delta_latency_ms"],
        ].round(4)
    )
    density = pd.read_csv(Path(output) / DENSITY_REPORT_FILE)
    if not density.empty:
        display(density.round(6))
    hypothesis = pd.read_csv(Path(output) / HYPOTHESIS_FILE)
    if hypothesis.empty:
        print(
            "Clear/fog hypothesis untested: this run has no labeled matched benchmark."
        )
        return
    print("Positive delta means density helped detection; negative means it hurt.")
    display(hypothesis.round(4))
    display(pd.read_csv(Path(output) / HYPOTHESIS_SUMMARY_FILE).round(4))


def display_training_and_data_health(output):
    print(
        "Checkpoint health: best_epoch is the evaluated checkpoint; "
        "epochs_trained is total fitted epochs."
    )
    display(pd.read_csv(Path(output) / TRAINING_HEALTH_FILE))
    print("Evaluation sample counts and labeled object counts:")
    display(pd.read_csv(Path(output) / DATASET_AUDIT_FILE))


def display_study_result_tables(results, output):
    display_detection_accuracy_table(results)
    display_inference_latency_table(results)
    display_fog_hypothesis_tables(output)
    display_training_and_data_health(output)


def display_saved_comparison_figures(output, results):
    if results.empty:
        return
    for pattern in (
        "*-comparison.png",
        "*-balanced.png",
        "*-deltas.png",
        "*-classes.png",
        "*-density.png",
        "*-loss.png",
        "*-conditions.png",
        "*-effects.png",
        "*-condition-classes.png",
        "*-precision-recall.png",
        DENSITY_DIAGNOSTIC_FIGURE,
        DATASET_AUDIT_FIGURE,
    ):
        for path in sorted((Path(output) / FIGURE_DIRECTORY).glob(pattern)):
            display(PreviewImage(filename=str(path), width=1000))


def export_checkpoints_and_reports():
    """Package the configured run and its recorded dependency versions."""
    if not RESOLVED_PATHS["run"].exists():
        raise FileNotFoundError(f"No run directory: {RESOLVED_PATHS['run']}")
    if RESOLVED_PATHS["packages"].exists():
        shutil.copy2(
            RESOLVED_PATHS["packages"],
            RESOLVED_PATHS["run"] / SETTINGS["output"]["packages_file"],
        )
    archive_path = shutil.make_archive(
        str(RESOLVED_PATHS["export"]), "zip", str(RESOLVED_PATHS["run"])
    )

    return archive_path

### Resolve storage and show the selected run

In [ ]:
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / ".ultralytics")
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

TRAINING_CONFIGURATION, RESOLVED_PATHS = resolve_study_configuration(SETTINGS)
display(
    pd.DataFrame(
        {
            "Setting": [
                "Profile",
                "Initialization",
                "Data source",
                "Output directory",
                "Models",
                "Arms",
                "Seeds",
                "Detector epochs",
                "Fog-density predictor",
                "Batch contract",
            ],
            "Value": [
                SETTINGS["profile"]["name"],
                SETTINGS["profile"]["initialization"],
                SETTINGS["data"]["location"],
                TRAINING_CONFIGURATION["output"],
                ", ".join(TRAINING_CONFIGURATION["checkpoints"]),
                ", ".join(TRAINING_CONFIGURATION["arms"]),
                TRAINING_CONFIGURATION["seeds"],
                TRAINING_CONFIGURATION["epochs"],
                TRAINING_CONFIGURATION.get("density_checkpoint")
                or "saved full-profile model",
                f"micro={TRAINING_CONFIGURATION['batch_size']} / accumulate={TRAINING_CONFIGURATION['accumulate']} / effective={TRAINING_CONFIGURATION['effective_batch_size']}",
            ],
        }
    )
)
print(
    "Run behavior: load data, restore or train the fog-density model, then resume detector checkpoints."
)

In [ ]:
# fmt: off
#@title Fog-density model implementation (collapsed)
# fmt: on
class FogDensityRegressor(nn.Module):
    """Small image-to-beta regressor used as the frozen fog predictor."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2, 2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1),
        )

    def forward(self, images):
        return (
            self.features(
                F.interpolate(
                    images,
                    (DENSITY_INPUT_SIZE, DENSITY_INPUT_SIZE),
                    mode="bilinear",
                    align_corners=False,
                )
            ).sigmoid()
            * BETA_SCALE
        )


def get_fog_density(images, predictor=None):
    """Estimate beta for a batch with the frozen image-to-density predictor."""
    predictor = predictor if predictor is not None else FOG_DENSITY_MODEL
    if predictor is None:
        raise ValueError("Run the fog-density model setup cell first.")
    with torch.inference_mode():
        return predictor(images)


def resolve_density_predictor_checkpoint(configuration, seed=None):
    """Return where this run saves or restores the fog-density model."""
    checkpoint = configuration.get("density_checkpoint")
    if checkpoint:
        return Path(checkpoint)
    seed = seed if seed is not None else configuration.get("seeds", [DEFAULT_SEED])[0]
    source = configuration.get("evaluation_source", configuration["output"])
    return Path(source) / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT


def load_fog_density_model(checkpoint_path, device):
    """Load a saved beta model once, freeze it, and return it for inference."""
    global FOG_DENSITY_MODEL, FOG_DENSITY_MODEL_KEY
    checkpoint_path = Path(checkpoint_path).resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Fog predictor not found: {checkpoint_path}. Set density_predictor.checkpoint to a saved {BEST_CHECKPOINT}."
        )
    stamp = checkpoint_path.stat()
    key = (str(checkpoint_path), device, stamp.st_size, stamp.st_mtime_ns)
    if FOG_DENSITY_MODEL is not None and FOG_DENSITY_MODEL_KEY == key:
        return FOG_DENSITY_MODEL
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model = FogDensityRegressor().to(device)
    model.load_state_dict(checkpoint["model"])
    FOG_DENSITY_MODEL = model.eval().requires_grad_(False)
    FOG_DENSITY_MODEL_KEY = key
    return FOG_DENSITY_MODEL

## 3. Prepare the study data

Prepare the same full dataset for every profile. Training selects the profile subset after all annotations are loaded.

In [ ]:
prepare_fog_data()

## 4. Load the datasets

nuscenes_data holds labeled nuScenes detector images. fog_density_rows holds Foggy Cityscapes images and beta labels for fitting the frozen predictor. The names keep detector data separate from density supervision.

### nuScenes detector data

In [ ]:
nuscenes_data = load_study_manifest(TRAINING_CONFIGURATION["manifest"])

### Foggy Cityscapes beta data

Load all Foggy Cityscapes beta images for the density predictor. The same fog images receive object boxes below for detector training; beta-only rows stay separate from detector labels.


In [ ]:
density_checkpoint = TRAINING_CONFIGURATION.get("density_checkpoint")
density_checkpoint_exists = density_checkpoint and Path(density_checkpoint).is_file()

In [ ]:
fog_density_rows = load_study_manifest(
    TRAINING_CONFIGURATION["city_manifest"],
    normalize_timestamps=not density_checkpoint_exists,
)

In [ ]:
fog_density_training_data = get_fog_model_training_data(fog_density_rows)

### Add labeled fog and clear Cityscapes detector examples

`prepare_cityscapes_data()` unpacks the uploaded `Cityscape-Dataset.zip` from `MyDrive/fog-study/` into `cityscapes-labeled/leftImg8bit/` and `cityscapes-labeled/gtFine/`. It skips extraction after the completion marker is written. The notebook then uses these instance masks to label Foggy Cityscapes images and pairs them with clear images. Keep the same scene split as the fog-density predictor: its training scenes stay in detector training, its held-out scenes stay in detector validation, and the official Cityscapes validation split is detector test data.


In [ ]:
prepare_cityscapes_data()
labeled_cityscapes_data = load_labeled_cityscapes_data(fog_density_rows)
run_identity_rows = nuscenes_data + labeled_cityscapes_data
display_dataset_split_summary(run_identity_rows, TRAINING_CONFIGURATION)

### Fit or resume the fog-density model

The model estimates the fog-density number attached to each labeled fog image (called beta in the dataset). This step always calls the fit function: it restores saved weights and epoch progress when available, starts from default weights otherwise, and returns the best model frozen for detector comparisons.


In [ ]:
fog_checkpoint = resolve_density_predictor_checkpoint(TRAINING_CONFIGURATION)
fog_predictor = fit_fog_density_model(
    fog_density_training_data,
    TRAINING_CONFIGURATION,
    fog_checkpoint,
    SETTINGS["density_predictor"]["seed"],
)

## 5. Train and validate the detector comparisons

Both detector arms train on the same nuScenes images and labeled Foggy Cityscapes scenes, including matched clear/fog examples. The RGB only control receives no density signal; the predicted-density arm receives the frozen predicted fog-density value for each image. YOLO11n, YOLO11m, and DETR each train and validate with both arms. Validation uses held-out Cityscapes scenes and the configured nuScenes split; final test uses official Cityscapes validation scenes and the nuScenes test split.


### Check inputs and prepare this profile

In [ ]:
detector_partitions = prepare_detector_study(
    run_identity_rows, run_identity_rows, TRAINING_CONFIGURATION
)

### Saved training progress

The table shows cumulative epochs and image steps for every configured comparison. Each model/arm below resumes its own checkpoint.

In [ ]:
display(read_run_continuation_summary(TRAINING_CONFIGURATION))

#### YOLO11n: RGB control

In [ ]:
yolo11n_rgb_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    YOLO_SMALL,
    ARM_RGB,
)
display(yolo11n_rgb_status)

#### YOLO11n: with predicted fog density

In [ ]:
yolo11n_density_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    YOLO_SMALL,
    ARM_DENSITY,
)
display(yolo11n_density_status)

#### YOLO11m: RGB control

In [ ]:
yolo11m_rgb_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    YOLO_MEDIUM,
    ARM_RGB,
)
display(yolo11m_rgb_status)

#### YOLO11m: with predicted fog density

In [ ]:
yolo11m_density_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    YOLO_MEDIUM,
    ARM_DENSITY,
)
display(yolo11m_density_status)

#### DETR: RGB control

In [ ]:
detr_rgb_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    DETR_MODEL,
    ARM_RGB,
)
display(detr_rgb_status)

#### DETR: with predicted fog density

In [ ]:
detr_density_status = train_and_validate_detector(
    detector_partitions,
    run_identity_rows,
    TRAINING_CONFIGURATION,
    fog_predictor,
    DETR_MODEL,
    ARM_DENSITY,
)
display(detr_density_status)

### Optional A100 input-loading benchmark

In [ ]:
RUN_SYNTHETIC_BENCHMARK = False
run_optional_synthetic_benchmark(RUN_SYNTHETIC_BENCHMARK)

### Optional: compare saved detectors on matched clear and fog images

Run this cell independently after loading the notebook settings and saved study paths. It loads detector and beta checkpoints when needed, then displays the matched fog/clear tables and graphs. Labeled Foggy Cityscapes (`gtFine`) must be available at the configured path.

In [ ]:
labeled_clear_fog_data = load_labeled_clear_fog_data()
fog_results, fog_status = run_saved_fog_comparison(labeled_clear_fog_data)
display(fog_status)
display_study_result_tables(
    fog_results,
    RESOLVED_PATHS["output_root"] / SETTINGS["evaluation"]["output_name"],
)
display_saved_comparison_figures(
    RESOLVED_PATHS["output_root"] / SETTINGS["evaluation"]["output_name"],
    fog_results,
)

## 6. Review the outputs

These results show detection accuracy, classes, loss, and latency on the configured evaluation split. To see whether the fog input helps foggy images while affecting clear images, run the matched Foggy Cityscapes cell above and compare its clear and fog rows.

### Tables: accuracy, overhead, and the hypothesis

`fog-vs-clear.csv` records each matched training seed's clear delta, fog delta and fog advantage (fog delta minus clear delta). `hypothesis-summary.csv` reports means and seed spread; at least three independent training seeds are required for exploratory bootstrap intervals. A missing interval is not a finding of no effect. The old nuScenes results stay separate with their original evaluation split.

In [ ]:
results = generate_study_analytics(TRAINING_CONFIGURATION["output"], run_identity_rows)
status = read_run_continuation_summary(TRAINING_CONFIGURATION)
display(status)
display_study_result_tables(results, TRAINING_CONFIGURATION["output"])

### Detector-study graphs

These figures summarize the object-detection comparison, including accuracy, fog-level behavior, class results, and latency with predictor cost included.

- **Clear/fog bars and effect bars:** compare each model and arm; zero is no effect.
- **Fog-level curves:** see whether a benefit changes with density.
- **Class heatmaps:** compare object categories; blank cells have no labeled examples.
- **Accuracy versus latency:** compare detection quality and inference cost.
- **Density diagnostics:** compare estimated beta with known beta.
- **Training loss and data audit:** check optimization progress and sample coverage.

In [ ]:
display_saved_comparison_figures(TRAINING_CONFIGURATION["output"], results)

### Subset completion and checkpoint reuse

## 7. Export the run

The ZIP is saved beside the run directory in the configured output location. It includes reports, checkpoints and the installed package versions.


In [ ]:
archive_path = export_checkpoints_and_reports()
print("Saved:", archive_path)

### Interpret the comparison

The RGB-only detector receives a fog value of zero. The density-enabled detector receives the estimate from the shared, frozen fog model. The main report measures each model on the configured labeled test split. The separate matched benchmark evaluates each saved model on the same Cityscapes scenes in clear and fog conditions, so those paired results are the direct test of the fog hypothesis.

Reports and checkpoints stay in the selected profile output folder. Use `analyze` to redraw saved reports; the matched evaluation can also be run on its own after loading the notebook settings and saved model paths.